# Goblet Cell Analysis: BBKNN Integration + Wilcoxon Marker Finding

**Purpose**: Re-integrate Goblet cells by dataset using BBKNN, then identify marker genes

**Workflow**:
1. Load data and inspect dataset distribution
2. Filter small datasets (< threshold)
3. Re-run preprocessing (HVG selection)
4. BBKNN batch correction by dataset
5. Clustering and visualization
6. Wilcoxon differential expression
7. Export results and figures

**Author**: r2end  
**Date**: 2025-01-05

## Configuration

In [ ]:
# ============================================================================
# CONFIGURATION PARAMETERS
# ============================================================================

# File paths
INPUT_H5AD = "/home/h2048/data/R/0105/goblet_bbknn/goblet_bbknn_processed_20260106.h5ad"  # ← MODIFY THIS
OUTPUT_DIR = "/home/h2048/data/R/0105/goblet_bbknn/"  # ← MODIFY THIS
FIGURE_DIR = f"{OUTPUT_DIR}/figures/"
TABLE_DIR = f"{OUTPUT_DIR}/tables/"
MARKER_TABLE_DIR = f"{OUTPUT_DIR}/marker_tables/"

import os
os.makedirs(FIGURE_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(MARKER_TABLE_DIR, exist_ok=True)

# Dataset filtering
BATCH_KEY = 'dataset'           # Column name for dataset/batch
MIN_CELLS_PER_DATASET = 20      # Filter datasets with < N cells

# Preprocessing
N_TOP_GENES = 4000              # Number of highly variable genes
N_PCS = 50                      # Number of PCs for dimensionality reduction

# BBKNN parameters
BBKNN_NEIGHBORS_WITHIN_BATCH = 4  # 3-5 for small batches, 5-10 for large batches
BBKNN_N_PCS = 25                  # Should match N_PCS above
BBKNN_TRIM = 30  # ← 添加这一行！可选值: None, 整数 (e.g., 10, 20)
# Clustering
LEIDEN_RESOLUTION = 1         # Adjust based on desired granularity

# Differential expression
MIN_LOGFC = 0.25                # Minimum log2 fold change
MIN_PCT = 0.1                   # Minimum expression percentage
TOP_N_MARKERS = 50              # Top markers per cluster

# Visualization
FIGURE_DPI = 300
FIGURE_FORMAT = 'pdf'
UMAP_SIZE = 3

# Performance
N_JOBS = 8

print("✓ Configuration loaded")

## Imports & Setup

In [ ]:
# ============================================================================
# IMPORTS
# ============================================================================

import scanpy as sc
import scanpy.external as sce
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import sparse
import warnings
warnings.filterwarnings('ignore')

# Scanpy settings
sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=FIGURE_DPI, facecolor='white', frameon=False)
sc.settings.n_jobs = N_JOBS

# Create output directories
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
fig_dir = output_dir / "figures"
fig_dir.mkdir(exist_ok=True)

print("="*80)
print("Goblet Cell BBKNN + Marker Analysis")
print("="*80)
print(f"Output directory: {output_dir}")

## Step 1: Load Data & Inspect Datasets

In [ ]:
# ============================================================================
# STEP 1: DATA LOADING & DATASET INSPECTION
# ============================================================================

print("\n" + "="*80)
print("STEP 1: DATA LOADING & DATASET INSPECTION")
print("="*80)

print(f"\nLoading: {INPUT_H5AD}")
adata = sc.read_h5ad(INPUT_H5AD)

print(f"\nData dimensions:")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")

# Check batch key exists
if BATCH_KEY not in adata.obs.columns:
    raise ValueError(f"Batch key '{BATCH_KEY}' not found in adata.obs!")

# Dataset distribution
print(f"\nDataset distribution ({BATCH_KEY}):")
dataset_counts = adata.obs[BATCH_KEY].value_counts().sort_values(ascending=False)
print(f"  Total datasets: {len(dataset_counts)}")
print(f"\n  Dataset sizes:")
for dataset, count in dataset_counts.items():
    print(f"    {dataset}: {count:,} cells ({count/adata.n_obs*100:.1f}%)")

# Identify small datasets
small_datasets = dataset_counts[dataset_counts < MIN_CELLS_PER_DATASET]
if len(small_datasets) > 0:
    print(f"\n  ⚠️  Small datasets (< {MIN_CELLS_PER_DATASET} cells): {len(small_datasets)}")
    for dataset, count in small_datasets.items():
        print(f"      {dataset}: {count} cells (will be removed)")
else:
    print(f"\n  ✓ No small datasets to filter")

In [ ]:
# ============================================================================
# FILTER LOW-QUALITY CLUSTERS AND COMPLETE RE-NORMALIZATION
# ============================================================================

print("\n" + "="*80)
print("FILTERING LOW-QUALITY CLUSTERS + COMPLETE RE-NORMALIZATION")
print("="*80)

# ============================================================================
# STEP 1: FILTER CLUSTERS
# ============================================================================

# Identify clusters to remove
clusters_to_remove = ['27', '31']
print(f"\nClusters to remove: {clusters_to_remove}")
print(f"  Cluster 27: Hemoglobin-positive (potential RBC contamination)")
print(f"  Cluster 31: Low-quality/doublet cells")

# Check cell counts before filtering
print(f"\nBefore filtering:")
n_cells_before = adata.n_obs
print(f"  Total cells: {n_cells_before:,}")
cells_to_remove = adata.obs['leiden'].isin(clusters_to_remove).sum()
print(f"  Cells in clusters {clusters_to_remove}: {cells_to_remove}")

# ============================================================================
# STEP 2: CLEAN UP OLD COMPUTED RESULTS
# ============================================================================
print("\nCleaning up old computed results...")

# Remove all computed structures to prevent TraitError
if 'neighbors' in adata.uns:
    del adata.uns['neighbors']
    print("  ✓ Removed neighbors")

if 'umap' in adata.obsm:
    del adata.obsm['umap']
    print("  ✓ Removed UMAP")

if 'dendrogram_leiden' in adata.uns:
    del adata.uns['dendrogram_leiden']
    print("  ✓ Removed dendrogram")

if 'rank_genes_wilcox' in adata.uns:
    del adata.uns['rank_genes_wilcox']
    print("  ✓ Removed old marker results")

if 'connectivities' in adata.obsp:
    del adata.obsp['connectivities']
if 'distances' in adata.obsp:
    del adata.obsp['distances']
    print("  ✓ Removed connectivity matrices")

# ============================================================================
# STEP 3: FILTER CELLS
# ============================================================================
print("\nFiltering cells...")
adata = adata[~adata.obs['leiden'].isin(clusters_to_remove)].copy()

# CRITICAL: Reset categorical to remove unused categories
adata.obs['leiden'] = adata.obs['leiden'].cat.remove_unused_categories()
print("  ✓ Reset leiden categories")

print(f"\nAfter filtering:")
print(f"  Total cells: {adata.n_obs:,}")
print(f"  Remaining clusters: {sorted(adata.obs['leiden'].unique())}")
print(f"  Cells removed: {n_cells_before - adata.n_obs} ({100*(n_cells_before - adata.n_obs)/n_cells_before:.2f}%)")

# Force garbage collection
import gc
gc.collect()
print("  ✓ Memory cleanup")

# ============================================================================
# STEP 4: RESTORE RAW COUNTS AND RE-NORMALIZE
# ============================================================================

print("\n" + "="*80)
print("RE-NORMALIZATION FROM RAW COUNTS")
print("="*80)

print("\nStep 1: Restoring original counts from .raw...")
if adata.raw is not None:
    # Create new AnnData with raw counts
    adata_renorm = sc.AnnData(
        X=adata.raw.X.copy(),
        obs=adata.obs.copy(),
        var=adata.raw.var.copy()
    )
    print(f"  ✓ Loaded {adata_renorm.n_obs} cells × {adata_renorm.n_vars} genes")
else:
    print("  ⚠️  Warning: .raw not found, using current .X")
    adata_renorm = adata.copy()

# Re-calculate QC metrics
print("\nStep 2: Re-calculating QC metrics...")
sc.pp.calculate_qc_metrics(
    adata_renorm,
    qc_vars=['mt', 'ribo'] if 'mt' in adata_renorm.var.columns else [],
    percent_top=None,
    log1p=False,
    inplace=True
)

print("\nQC summary after filtering:")
print(f"  Mean counts: {adata_renorm.obs['total_counts'].mean():.0f}")
print(f"  Mean genes: {adata_renorm.obs['n_genes_by_counts'].mean():.0f}")
if 'pct_counts_mt' in adata_renorm.obs.columns:
    print(f"  Mean MT%: {adata_renorm.obs['pct_counts_mt'].mean():.2f}%")

# Re-normalize
print("\nStep 3: Re-normalizing...")
sc.pp.normalize_total(adata_renorm, target_sum=1e4)
sc.pp.log1p(adata_renorm)
print("  ✓ Normalization complete")

# Save to layers
adata_renorm.layers['log1p'] = adata_renorm.X.copy()
print("  ✓ Saved to layers['log1p']")

# Update .raw with filtered counts
print("\nStep 4: Updating .raw...")
adata_renorm.raw = sc.AnnData(
    X=adata.raw.X.copy(),
    obs=adata_renorm.obs.copy(),
    var=adata_renorm.var.copy()
)
print("  ✓ .raw updated with filtered counts")

# Replace original adata
adata = adata_renorm
del adata_renorm
gc.collect()

print("\n✓ Re-normalization complete!")

## Step 2: Filter Small Datasets

In [ ]:
# ============================================================================
# STEP 2: FILTER SMALL DATASETS
# ============================================================================

print("\n" + "="*80)
print("STEP 2: FILTER SMALL DATASETS")
print("="*80)

n_cells_before = adata.n_obs

# Filter small datasets
dataset_counts = adata.obs[BATCH_KEY].value_counts()
valid_datasets = dataset_counts[dataset_counts >= MIN_CELLS_PER_DATASET].index

adata = adata[adata.obs[BATCH_KEY].isin(valid_datasets)].copy()

n_cells_after = adata.n_obs
n_cells_removed = n_cells_before - n_cells_after

print(f"\nFiltering results:")
print(f"  Cells before: {n_cells_before:,}")
print(f"  Cells after: {n_cells_after:,}")
print(f"  Cells removed: {n_cells_removed:,} ({n_cells_removed/n_cells_before*100:.1f}%)")
print(f"  Remaining datasets: {adata.obs[BATCH_KEY].nunique()}")

# Update dataset distribution
print(f"\nRemaining dataset distribution:")
dataset_counts_filtered = adata.obs[BATCH_KEY].value_counts().sort_values(ascending=False)
for dataset, count in dataset_counts_filtered.items():
    print(f"  {dataset}: {count:,} cells")

## Step 3: Data State Check & Preparation

In [ ]:
# ============================================================================
# STEP 3: DATA STATE CHECK & PREPARATION
# ============================================================================

print("\n" + "="*80)
print("STEP 3: DATA STATE VERIFICATION")
print("="*80)

# Check data state
print(f"\nChecking data structure...")

# Check .X
X_min, X_max = adata.X.min(), adata.X.max()
X_mean = adata.X.mean()
print(f"\n.X matrix:")
print(f"  Type: {type(adata.X)}")
print(f"  Range: [{X_min:.4f}, {X_max:.4f}]")
print(f"  Mean: {X_mean:.4f}")

# Check layers
if 'counts' in adata.layers:
    print(f"\n✓ .layers['counts'] exists")
    HAS_COUNTS = True
else:
    print(f"\n⚠️  .layers['counts'] not found")
    print(f"   Will assume .X contains counts if in appropriate range")
    HAS_COUNTS = False

# Prepare counts layer if needed
if not HAS_COUNTS:
    if X_max > 100:  # Likely counts
        print(f"\n  Saving .X to .layers['counts']")
        adata.layers['counts'] = adata.X.copy()
        HAS_COUNTS = True
    else:
        raise ValueError(".X appears to be normalized but no counts layer found!")

In [ ]:
# ============================================================================
# STEP 7.5: FILTER UNWANTED GENE CATEGORIES
# ============================================================================
# Insert this cell AFTER Step 7 (Visualization) and BEFORE Step 8 (Wilcoxon DE)

print("\n" + "="*80)
print("STEP 7.5: FILTER UNWANTED GENE CATEGORIES")
print("="*80)

print("\nFiltering genes to improve marker quality...")
print("Will remove: MT, ribosomal, histone, pseudogenes, ENSG, unannotated transcripts")

# ===== Configuration =====
REMOVE_MT = True
REMOVE_RIBO = True
REMOVE_HISTONE = True
REMOVE_PSEUDOGENES = True
REMOVE_ENSG = True
REMOVE_UNANNOTATED = True

# ===== Get all gene names from adata.raw =====
if adata.raw is None:
    print("\n⚠️  Warning: adata.raw is None, will filter adata.var instead")
    all_genes = adata.var_names.tolist()
    use_raw = False
else:
    all_genes = adata.raw.var_names.tolist()
    use_raw = True

print(f"\nTotal genes before filtering: {len(all_genes):,}")

# ===== Initialize list of genes to remove =====
genes_to_remove = []

# ===== 1. Mitochondrial genes (MT-) =====
if REMOVE_MT:
    mt_genes = [g for g in all_genes if g.startswith('MT-')]
    genes_to_remove.extend(mt_genes)
    print(f"\n  1. Mitochondrial genes (MT-): {len(mt_genes)} genes")
    if len(mt_genes) > 0:
        print(f"     Examples: {', '.join(mt_genes[:5])}")

# ===== 2. Ribosomal genes (RPS, RPL, MRPS, MRPL) =====
if REMOVE_RIBO:
    import re
    ribo_pattern = re.compile(r'^(RPS|RPL|MRPS|MRPL)')
    ribo_genes = [g for g in all_genes if ribo_pattern.match(g)]
    genes_to_remove.extend(ribo_genes)
    print(f"  2. Ribosomal genes (RPS/RPL/MRPS/MRPL): {len(ribo_genes)} genes")
    if len(ribo_genes) > 0:
        print(f"     Examples: {', '.join(ribo_genes[:5])}")

# ===== 3. Histone genes (H1, H2A, H2B, H3, H4, HIST) =====
if REMOVE_HISTONE:
    histone_pattern = re.compile(r'^(H1|H2A|H2B|H3|H4|HIST)')
    histone_genes = [g for g in all_genes if histone_pattern.match(g)]
    genes_to_remove.extend(histone_genes)
    print(f"  3. Histone genes (H1/H2A/H2B/H3/H4/HIST): {len(histone_genes)} genes")
    if len(histone_genes) > 0:
        print(f"     Examples: {', '.join(histone_genes[:5])}")

# ===== 4. Pseudogenes (e.g., RPS29P1, RPL10P9) =====
if REMOVE_PSEUDOGENES:
    # Pattern: RPS/RPL/MRPS/MRPL + digits + P + digits
    pseudo_pattern = re.compile(r'^(RPS|RPL|MRPS|MRPL)[0-9]+P[0-9]+$')
    pseudo_genes = [g for g in all_genes if pseudo_pattern.match(g)]
    genes_to_remove.extend(pseudo_genes)
    print(f"  4. Pseudogenes (*P*): {len(pseudo_genes)} genes")
    if len(pseudo_genes) > 0:
        print(f"     Examples: {', '.join(pseudo_genes[:5])}")

# ===== 5. ENSG unannotated genes =====
if REMOVE_ENSG:
    ensg_pattern = re.compile(r'^ENSG[0-9]+')
    ensg_genes = [g for g in all_genes if ensg_pattern.match(g)]
    genes_to_remove.extend(ensg_genes)
    print(f"  5. ENSG unannotated genes: {len(ensg_genes)} genes")
    if len(ensg_genes) > 0:
        print(f"     Examples: {', '.join(ensg_genes[:5])}")

# ===== 6. Unannotated transcripts =====
if REMOVE_UNANNOTATED:
    # Pattern includes:
    # - AC/AL/AP/BX/Z followed by digits and dot
    # - RP followed by digits and dash
    # - CTD-/CTB-/CTC-
    # - LINC followed by digits
    # - Ending with -AS + digits (antisense)
    # - Ending with -OT + digits (overlapping transcript)
    # - Starting with LOC + digits
    unannotated_pattern = re.compile(
        r'^(AC|AL|AP|BX|Z)[0-9]+\.|'
        r'^RP[0-9]+-|'
        r'^CTD-|^CTB-|^CTC-|'
        r'^LINC[0-9]+|'
        r'-AS[0-9]+$|'
        r'-OT[0-9]+$|'
        r'^LOC[0-9]+'
    )
    unannotated_genes = [g for g in all_genes if unannotated_pattern.search(g)]
    genes_to_remove.extend(unannotated_genes)
    print(f"  6. Unannotated transcripts (AC/AL/RP/CTD/LINC/LOC/etc): {len(unannotated_genes)} genes")
    if len(unannotated_genes) > 0:
        print(f"     Examples: {', '.join(unannotated_genes[:5])}")

# ===== Remove duplicates =====
genes_to_remove = list(set(genes_to_remove))
print(f"\n{'─'*80}")
print(f"Total unique genes to remove: {len(genes_to_remove):,}")

# ===== Determine genes to keep =====
genes_to_keep = [g for g in all_genes if g not in genes_to_remove]
print(f"Genes to keep: {len(genes_to_keep):,}")
print(f"Percentage retained: {len(genes_to_keep)/len(all_genes)*100:.1f}%")

# ===== Filter adata.raw =====
if use_raw:
    print(f"\nFiltering adata.raw...")
    # IMPORTANT: Must convert adata.raw to AnnData, filter, then reassign
    # Direct slicing of adata.raw does not work
    
    # Step 1: Convert raw to full AnnData object
    raw_adata = adata.raw.to_adata()
    
    # Step 2: Filter genes
    raw_adata_filtered = raw_adata[:, genes_to_keep].copy()
    
    # Step 3: Reassign to adata.raw
    adata.raw = raw_adata_filtered
    
    print(f"  ✓ adata.raw filtered: {adata.raw.n_vars:,} genes remain")
    
    # Clean up temporary objects
    del raw_adata, raw_adata_filtered
    
else:
    print(f"\nFiltering adata.var...")
    adata = adata[:, genes_to_keep].copy()
    print(f"  ✓ adata filtered: {adata.n_vars:,} genes remain")

# ===== Summary of removed gene categories =====
print(f"\n{'─'*80}")
print("Summary of removed gene categories:")
if REMOVE_MT:
    print(f"  ✓ Mitochondrial genes (MT-)")
if REMOVE_RIBO:
    print(f"  ✓ Ribosomal genes (RPS/RPL/MRPS/MRPL)")
if REMOVE_HISTONE:
    print(f"  ✓ Histone genes (H1/H2A/H2B/H3/H4/HIST)")
if REMOVE_PSEUDOGENES:
    print(f"  ✓ Pseudogenes (*P*)")
if REMOVE_ENSG:
    print(f"  ✓ ENSG unannotated genes")
if REMOVE_UNANNOTATED:
    print(f"  ✓ Unannotated transcripts (AC/AL/RP/CTD/LINC/LOC/etc)")

print("\n✓ Gene filtering complete")
print("  → Subsequent differential expression will use filtered gene set")

In [ ]:
# 选择 counts 来源：优先用 layers['counts']，否则用 X（如果 X 是 lognorm，不影响 score，但会影响 hb_fraction）
COUNTS = "counts" if "counts" in adata.layers else None
print("COUNTS layer:", COUNTS)

var_names = adata.raw.var_names if adata.raw is not None else adata.var_names
def present(genes): 
    return [g for g in genes if g in var_names]

# 上皮（更“结构性”的上皮 marker，避免把 IL8/SAA 当成免疫）
EPI = present(["EPCAM","KRT8","KRT18","KRT19","KRT7","MUC1"])
MYE = present(["PTPRC","LST1","TYROBP","FCER1G","CTSS","MS4A7","FCGR3A","LGALS3"])
HB  = present(["HBB","HBA1","HBA2","HBD"])

print("EPI genes:", EPI)
print("MYE genes:", MYE)
print("HB genes:", HB)

# score_genes 默认用 adata.X（通常是 lognorm），用于“双高”识别足够稳
sc.tl.score_genes(adata, gene_list=EPI, score_name="score_epi", use_raw=(adata.raw is not None))
sc.tl.score_genes(adata, gene_list=MYE, score_name="score_mye", use_raw=(adata.raw is not None))

# hb_fraction：尽量用 counts
if len(HB) > 0:
    X = adata.layers[COUNTS] if COUNTS is not None else adata.X
    # 取 HB 基因列
    hb_idx = [list(adata.var_names).index(g) for g in HB if g in adata.var_names] if (adata.raw is None) else [list(adata.raw.var_names).index(g) for g in HB]
    # 如果用了 raw，需要从 raw 取 counts；raw 通常不带 layers，所以这里建议你确保 HB genes 在 adata.var_names 并用 counts layer
    if COUNTS is None:
        print("WARNING: no counts layer; hb_fraction will be approximate if X is normalized/log.")
    import scipy.sparse as sp
    Xmat = X if not sp.issparse(X) else X.tocsr()
    total = np.array(Xmat.sum(axis=1)).ravel()
    # hb sum：仅当 HB genes 在 adata.var_names 才严格
    if all(g in adata.var_names for g in HB) and COUNTS is not None:
        hb_cols = [adata.var_names.get_loc(g) for g in HB]
        hb_sum = np.array(Xmat[:, hb_cols].sum(axis=1)).ravel()
        adata.obs["hb_fraction"] = np.divide(hb_sum, np.maximum(total, 1), dtype=float)
    else:
        adata.obs["hb_fraction"] = np.nan
else:
    adata.obs["hb_fraction"] = np.nan

adata.obs[["score_epi","score_mye","hb_fraction"]].describe()

import re
import numpy as np
import pandas as pd
import scanpy as sc

print("layers:", list(adata.layers.keys()))
print("obs columns (n=%d):" % adata.obs.shape[1])
print(sorted(adata.obs.columns.tolist())[:50], " ...")

# 找 DecontX contamination 相关列（大小写/下划线不确定）
pat = re.compile(r"(decontx|contam)", re.IGNORECASE)
cand = [c for c in adata.obs.columns if pat.search(c)]
print("DecontX-like columns:", cand)


In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

# ---- sanity checks ----
assert "counts" in adata.layers, "No counts layer found."
X = adata.layers["counts"]
X = X.tocsr() if sp.issparse(X) else np.asarray(X)

HB = ["HBB","HBA1","HBA2","HBD"]
hb_idx = adata.var_names.get_indexer(HB)

missing = [g for g,i in zip(HB, hb_idx) if i < 0]
present = [g for g,i in zip(HB, hb_idx) if i >= 0]
print("HB present:", present)
print("HB missing:", missing)

hb_idx = hb_idx[hb_idx >= 0]
if len(hb_idx) == 0:
    raise ValueError("No HB genes found in adata.var_names; cannot compute hb_fraction.")

# ---- compute totals ----
total_umi = np.array(X.sum(axis=1)).ravel() if sp.issparse(X) else X.sum(axis=1)
hb_umi = np.array(X[:, hb_idx].sum(axis=1)).ravel() if sp.issparse(X) else X[:, hb_idx].sum(axis=1)

adata.obs["total_umi_counts"] = total_umi.astype(np.float64)
adata.obs["hb_umi"] = hb_umi.astype(np.float64)
adata.obs["hb_fraction"] = adata.obs["hb_umi"] / np.maximum(adata.obs["total_umi_counts"], 1.0)

adata.obs[["hb_umi","hb_fraction","total_umi_counts"]].describe()

s = adata.obs["hb_fraction"].astype(float)

qs = s.quantile([0.90, 0.95, 0.99, 0.995, 0.999])
print(qs)

# 经验：上皮子集里 hb_fraction > 0.01 (1%) 通常已经很可疑
# 更保守：取 max(1%, 99.5分位)
HB_MAX = float(max(0.01, qs.loc[0.995]))
print("HB_MAX =", HB_MAX)

import scipy.sparse as sp
import numpy as np

X = adata.layers["counts"]
X = X.tocsr() if sp.issparse(X) else np.asarray(X)

def gene_counts(g):
    j = adata.var_names.get_loc(g) if g in adata.var_names else None
    if j is None:
        return np.zeros(adata.n_obs, dtype=np.float64)
    v = X[:, j]
    return np.array(v.toarray()).ravel() if sp.issparse(X) else v

ptprc = gene_counts("PTPRC")
lst1  = gene_counts("LST1")
tyrobp= gene_counts("TYROBP")

adata.obs["PTPRC_counts"] = ptprc
adata.obs["LST1_counts"]  = lst1
adata.obs["TYROBP_counts"]= tyrobp

# “髓系阳性”——在上皮子集里通常可直接删除（你也可以改成 >1 更保守）
adata.obs["flag_mye_pos"] = (adata.obs["PTPRC_counts"] > 0) | (adata.obs["LST1_counts"] > 0) | (adata.obs["TYROBP_counts"] > 0)

# “epi+mye 双高” doublet-like（用分位数更稳）
q_epi = adata.obs["score_epi"].quantile(0.85)
q_mye = adata.obs["score_mye"].quantile(0.85)
adata.obs["flag_epi_mye_doubletlike"] = (adata.obs["score_epi"] >= q_epi) & (adata.obs["score_mye"] >= q_mye)

print("mye_pos:", int(adata.obs["flag_mye_pos"].sum()))
print("epi+mye doublet-like:", int(adata.obs["flag_epi_mye_doubletlike"].sum()))

import pandas as pd
import numpy as np

# 1) DecontX contamination
DECONTX_MAX = 0.25  # 常用 0.20~0.30；你这类子集建议先 0.25
cont = pd.to_numeric(adata.obs["decontX_contamination"], errors="coerce")
mask_decontx = (cont <= DECONTX_MAX) | cont.isna()

# 2) HB extreme
HB_MAX = float(max(0.01, adata.obs["hb_fraction"].quantile(0.995)))  # 只杀极端
mask_hb = (adata.obs["hb_fraction"] <= HB_MAX) | adata.obs["hb_fraction"].isna()

# 3) myeloid-positive removal (上皮子集中通常建议直接去掉)
mask_mye = ~adata.obs["flag_mye_pos"]

# 4) epi+mye doublet-like removal（更像上皮+髓系双细胞）
mask_doubletlike = ~adata.obs["flag_epi_mye_doubletlike"]

mask_keep = mask_decontx & mask_hb & mask_mye & mask_doubletlike
print("Remove DecontX:", int((~mask_decontx).sum()))
print("Remove HB:", int((~mask_hb).sum()))
print("Remove mye_pos:", int((~mask_mye).sum()))
print("Remove epi+mye doublet-like:", int((~mask_doubletlike).sum()))
print("Total keep:", int(mask_keep.sum()), "/", adata.n_obs)

adata = adata[mask_keep].copy()


## Step 4: Preprocessing (HVG + PCA)

In [ ]:
# ============================================================================
# STEP 4: PREPROCESSING
# ============================================================================

print("\n" + "="*80)
print("STEP 4: PREPROCESSING (HVG + PCA)")
print("="*80)

# Normalize and log-transform
print(f"\nNormalization...")
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print(f"  ✓ Log-normalized (target_sum=1e4)")

In [ ]:


# HVG selection (batch-aware)
print(f"\nHighly variable genes selection...")
print(f"  Method: batch-aware (flavor='seurat_v3')")
print(f"  Target genes: {N_TOP_GENES}")
print(f"  Batch key: {BATCH_KEY}")

try:
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=N_TOP_GENES,
        batch_key=BATCH_KEY,
        flavor='seurat_v3',
        subset=False
    )
    hvg_method = "batch-aware"
    print(f"  ✓ Batch-aware HVG completed")
except Exception as e:
    print(f"  ⚠️  Batch-aware HVG failed: {e}")
    print(f"  Falling back to non-batch-aware method...")
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=N_TOP_GENES,
        flavor='seurat_v3',
        subset=False
    )
    hvg_method = "non-batch-aware"
    print(f"  ✓ Non-batch-aware HVG completed")

n_hvg = adata.var['highly_variable'].sum()
print(f"  Selected HVGs: {n_hvg}")

# Save full gene data to .raw (for later marker analysis)
print(f"\nPreserving full gene data...")
adata.raw = adata.copy()
print(f"  ✓ adata.raw saved (all {adata.n_vars} genes)")

# Subset to HVGs for integration
adata = adata[:, adata.var['highly_variable']].copy()
print(f"  ✓ Subset to {adata.n_vars} HVGs for integration")

# PCA
print(f"\nPCA computation...")
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=N_PCS, svd_solver='arpack')
print(f"  ✓ PCA completed ({N_PCS} components)")

## Step 5: BBKNN Batch Correction

In [ ]:
# ============================================================================
# STEP 5: BBKNN BATCH CORRECTION
# ============================================================================

print("\n" + "="*80)
print("STEP 5: BBKNN BATCH CORRECTION")
print("="*80)

print(f"\nBBKNN parameters:")
print(f"  Batch key: {BATCH_KEY}")
print(f"  Neighbors within batch: {BBKNN_NEIGHBORS_WITHIN_BATCH}")
print(f"  Number of PCs: {BBKNN_N_PCS}")
print(f"  Trim: {BBKNN_TRIM}")  # ← 添加这一行显示trim参数
print(f"  Number of datasets: {adata.obs[BATCH_KEY].nunique()}")

print(f"\nRunning BBKNN...")
sce.pp.bbknn(
    adata,
    batch_key=BATCH_KEY,
    neighbors_within_batch=BBKNN_NEIGHBORS_WITHIN_BATCH,
    n_pcs=BBKNN_N_PCS,
    trim=BBKNN_TRIM  # ← 修改这一行，从trim=None改为trim=BBKNN_TRIM
)
print(f"  ✓ BBKNN completed")

# UMAP
print(f"\nComputing UMAP...")
sc.tl.umap(adata)
print(f"  ✓ UMAP completed")

## Step 6: Clustering

In [ ]:
# ============================================================================
# STEP 6: CLUSTERING
# ============================================================================

print("\n" + "="*80)
print("STEP 6: LEIDEN CLUSTERING")
print("="*80)

print(f"\nClustering parameters:")
print(f"  Method: Leiden")
print(f"  Resolution: {LEIDEN_RESOLUTION}")

sc.tl.leiden(adata, resolution=LEIDEN_RESOLUTION, key_added='leiden')

n_clusters = adata.obs['leiden'].nunique()
print(f"\n✓ Clustering completed: {n_clusters} clusters")

# Cluster distribution
print(f"\nCluster sizes:")
cluster_counts = adata.obs['leiden'].value_counts().sort_index()
for cluster, count in cluster_counts.items():
    print(f"  Cluster {cluster}: {count:,} cells ({count/adata.n_obs*100:.1f}%)")

## Step 7: Visualization

In [ ]:
# ============================================================================
# STEP 7: VISUALIZATION
# ============================================================================

print("\n" + "="*80)
print("STEP 7: VISUALIZATION")
print("="*80)

# UMAP by cluster
print(f"\nGenerating UMAP plots...")

fig, ax = plt.subplots(figsize=(8, 6))
sc.pl.umap(
    adata,
    color='leiden',
    ax=ax,
    show=False,
    legend_loc='on data',
    legend_fontsize=8,
    size=UMAP_SIZE,
    title='Goblet Cell Clusters (BBKNN-integrated)'
)
plt.tight_layout()
plt.savefig(fig_dir / f'01_umap_clusters.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: 01_umap_clusters.{FIGURE_FORMAT}")

# UMAP by dataset (batch mixing)
fig, ax = plt.subplots(figsize=(8, 6))
sc.pl.umap(
    adata,
    color=BATCH_KEY,
    ax=ax,
    show=False,
    size=UMAP_SIZE,
    title=f'Batch Mixing ({BATCH_KEY})'
)
plt.tight_layout()
plt.savefig(fig_dir / f'02_umap_batch.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: 02_umap_batch.{FIGURE_FORMAT}")

# Combined view
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sc.pl.umap(adata, color='leiden', ax=axes[0], show=False, legend_loc='on data', size=UMAP_SIZE)
sc.pl.umap(adata, color=BATCH_KEY, ax=axes[1], show=False, size=UMAP_SIZE)
axes[0].set_title('Clusters', fontsize=14, weight='bold')
axes[1].set_title('Dataset Batch', fontsize=14, weight='bold')
plt.tight_layout()
plt.savefig(fig_dir / f'03_umap_combined.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: 03_umap_combined.{FIGURE_FORMAT}")

## Step 8: Wilcoxon Differential Expression

In [ ]:
# ============================================================================
# STEP 8: WILCOXON DIFFERENTIAL EXPRESSION
# ============================================================================

print("\n" + "="*80)
print("STEP 8: DIFFERENTIAL EXPRESSION ANALYSIS")
print("="*80)

print(f"\nRunning Wilcoxon rank-sum test...")
print(f"  Method: One-vs-rest comparison")
print(f"  Using: adata.raw (all genes)")
print(f"  Top N genes per cluster: {TOP_N_MARKERS}")

# Run Wilcoxon test on full gene set
sc.tl.rank_genes_groups(
    adata,
    groupby='leiden',
    method='wilcoxon',
    use_raw=True,  # Use full gene set from adata.raw
    n_genes=TOP_N_MARKERS,
    key_added='rank_genes_wilcox'
)
print(f"  ✓ Differential expression completed")

## Step 9: Extract & Filter Markers

In [ ]:
# ============================================================================
# STEP 9: EXTRACT & FILTER MARKER GENES
# ============================================================================

print("\n" + "="*80)
print("STEP 9: EXTRACTING & FILTERING MARKERS")
print("="*80)

print(f"\nFiltering criteria:")
print(f"  - log2FC > {MIN_LOGFC}")
print(f"  - Adjusted p-value < 0.05")
print(f"  - Expression % in cluster > {MIN_PCT*100}%")

all_markers = []

for cluster in adata.obs['leiden'].cat.categories:
    # Get DE results
    cluster_result = sc.get.rank_genes_groups_df(
        adata,
        group=cluster,
        key='rank_genes_wilcox'
    )
    
    # Basic filtering
    cluster_result_filtered = cluster_result[
        (cluster_result['logfoldchanges'] > MIN_LOGFC) &
        (cluster_result['pvals_adj'] < 0.05)
    ].copy()
    
    if len(cluster_result_filtered) == 0:
        print(f"  Cluster {cluster}: No significant markers")
        continue
    
    # Calculate expression percentage
    cluster_cells = adata.raw.X[adata.obs['leiden'] == cluster]
    other_cells = adata.raw.X[adata.obs['leiden'] != cluster]
    
    pct_in = []
    pct_out = []
    
    for gene in cluster_result_filtered['names']:
        gene_idx = adata.raw.var_names.get_loc(gene)
        
        if sparse.issparse(cluster_cells):
            expr_in = cluster_cells[:, gene_idx].toarray().flatten()
            expr_out = other_cells[:, gene_idx].toarray().flatten()
        else:
            expr_in = cluster_cells[:, gene_idx].flatten()
            expr_out = other_cells[:, gene_idx].flatten()
        
        pct_in.append(np.sum(expr_in > 0) / len(expr_in))
        pct_out.append(np.sum(expr_out > 0) / len(expr_out))
    
    cluster_result_filtered['cluster'] = cluster
    cluster_result_filtered['pct_in_cluster'] = pct_in
    cluster_result_filtered['pct_out_cluster'] = pct_out
    
    # Final filtering
    cluster_result_filtered = cluster_result_filtered[
        cluster_result_filtered['pct_in_cluster'] > MIN_PCT
    ]
    
    n_markers = len(cluster_result_filtered)
    print(f"  Cluster {cluster}: {n_markers} markers")
    
    all_markers.append(cluster_result_filtered)

# Combine results
if len(all_markers) > 0:
    markers_df = pd.concat(all_markers, ignore_index=True)
    markers_df = markers_df.sort_values(['cluster', 'pvals_adj'])
    
    # Save results
    markers_df.to_csv(output_dir / "markers_wilcox_all.csv", index=False)
    print(f"\n✓ Total markers: {len(markers_df):,}")
    print(f"✓ Saved: markers_wilcox_all.csv")
    
    # Top 10 per cluster
    top_markers = markers_df.groupby('cluster').head(10)
    top_markers.to_csv(output_dir / "markers_wilcox_top10.csv", index=False)
    print(f"✓ Saved: markers_wilcox_top10.csv")
else:
    print("\n⚠️  No markers passed filtering!")
    markers_df = pd.DataFrame()

## Step 10: Marker Visualization

In [ ]:
# ============================================================================
# STEP 10: MARKER VISUALIZATION
# ============================================================================

if len(markers_df) > 0:
    print("\n" + "="*80)
    print("STEP 10: MARKER VISUALIZATION")
    print("="*80)
    
    # Heatmap
    print(f"\nGenerating heatmap (top 5 per cluster)...")
    try:
        fig = sc.pl.rank_genes_groups_heatmap(
            adata,
            n_genes=5,
            key='rank_genes_wilcox',
            use_raw=True,
            show=False,
            cmap='RdBu_r',
            figsize=(12, 10),
            vmin=-3,
            vmax=3,
            dendrogram=False
        )
        plt.savefig(fig_dir / f'04_marker_heatmap.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
        plt.show()
        print(f"  ✓ Saved: 04_marker_heatmap.{FIGURE_FORMAT}")
    except Exception as e:
        print(f"  ⚠️  Heatmap failed: {e}")
    
    # Dotplot - recalculate dendrogram or disable it
    print(f"\nGenerating dotplot (top 3 per cluster)...")
    try:
        # Option 1: Recalculate dendrogram to match current clusters
        print("  Recalculating dendrogram for current cluster structure...")
        sc.tl.dendrogram(adata, groupby='leiden')
        
        fig = sc.pl.rank_genes_groups_dotplot(
            adata,
            n_genes=3,
            key='rank_genes_wilcox',
            use_raw=True,
            show=False,
            figsize=(14, 6),
            dendrogram=True  # Now safe to use dendrogram
        )
        plt.savefig(fig_dir / f'05_marker_dotplot.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
        plt.show()
        print(f"  ✓ Saved: 05_marker_dotplot.{FIGURE_FORMAT}")
    except Exception as e:
        print(f"  ⚠️  Dotplot with dendrogram failed: {e}")
        print("  Retrying without dendrogram...")
        try:
            # Option 2: Disable dendrogram as fallback
            fig = sc.pl.rank_genes_groups_dotplot(
                adata,
                n_genes=3,
                key='rank_genes_wilcox',
                use_raw=True,
                show=False,
                figsize=(14, 6),
                dendrogram=False
            )
            plt.savefig(fig_dir / f'05_marker_dotplot.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
            plt.show()
            print(f"  ✓ Saved: 05_marker_dotplot.{FIGURE_FORMAT}")
        except Exception as e2:
            print(f"  ✗ Dotplot failed completely: {e2}")
    
    # Violin plot for top marker per cluster
    print(f"\nGenerating violin plots...")
    top1_per_cluster = markers_df.groupby('cluster').first().reset_index()
    top_genes = top1_per_cluster['names'].tolist()[:12]
    
    if len(top_genes) > 0:
        try:
            n_cols = 3
            n_rows = int(np.ceil(len(top_genes) / n_cols))
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows))
            axes = axes.flatten() if n_rows > 1 else [axes]
            
            for idx, gene in enumerate(top_genes):
                sc.pl.violin(adata, keys=gene, groupby='leiden',
                            ax=axes[idx], show=False, use_raw=True)
                axes[idx].set_title(f'{gene}', fontsize=10, weight='bold')
            
            for idx in range(len(top_genes), len(axes)):
                axes[idx].axis('off')
            
            plt.tight_layout()
            plt.savefig(fig_dir / f'06_marker_violin.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
            plt.show()
            print(f"  ✓ Saved: 06_marker_violin.{FIGURE_FORMAT}")
        except Exception as e:
            print(f"  ⚠️  Violin plot failed: {e}")

print("\n✓ Visualization complete")

## Step 11: Save Processed Data

In [ ]:
# ============================================================================
# STEP 11: SAVE PROCESSED DATA
# ============================================================================

print("\n" + "="*80)
print("STEP 11: SAVING PROCESSED DATA")
print("="*80)

output_h5ad = output_dir / "goblet_bbknn_semi_final_20260106.h5ad"

print(f"\nSaving AnnData object...")
print(f"  Output: {output_h5ad}")
print(f"  Contents:")
print(f"    - Cells: {adata.n_obs:,}")
print(f"    - HVGs: {adata.n_vars:,}")
print(f"    - Full genes in .raw: {adata.raw.n_vars:,}")
print(f"    - BBKNN neighbors: ✓")
print(f"    - UMAP: ✓")
print(f"    - Leiden clusters: ✓")
print(f"    - Wilcoxon DE results: ✓")

adata.write_h5ad(output_h5ad, compression='gzip')
print(f"\n✓ Data saved successfully")

## Summary

In [ ]:
# ============================================================================
# ANALYSIS SUMMARY
# ============================================================================

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)

print(f"\nOutput directory: {output_dir}")

print(f"\nGenerated files:")
print(f"  ├── goblet_bbknn_processed.h5ad (processed data)")
print(f"  ├── markers_wilcox_all.csv (all markers)")
print(f"  ├── markers_wilcox_top10.csv (top 10 per cluster)")
print(f"  └── figures/")
print(f"      ├── 01_umap_clusters.{FIGURE_FORMAT}")
print(f"      ├── 02_umap_batch.{FIGURE_FORMAT}")
print(f"      ├── 03_umap_combined.{FIGURE_FORMAT}")
print(f"      ├── 04_marker_heatmap.{FIGURE_FORMAT}")
print(f"      ├── 05_marker_dotplot.{FIGURE_FORMAT}")
print(f"      └── 06_marker_violin.{FIGURE_FORMAT}")

if len(markers_df) > 0:
    print(f"\nMarker summary by cluster:")
    for cluster in sorted(markers_df['cluster'].unique()):
        cluster_markers = markers_df[markers_df['cluster'] == cluster]
        top3 = cluster_markers.head(3)['names'].tolist()
        print(f"  Cluster {cluster}: {len(cluster_markers)} markers")
        print(f"    Top 3: {', '.join(top3)}")

print("\n" + "="*80)
print("Next steps:")
print("  1. Inspect UMAP for batch mixing quality")
print("  2. Review marker genes for biological relevance")
print("  3. Annotate clusters based on markers")
print("  4. Consider subclustering if needed")
print("="*80)

In [ ]:
# In[ ]: Cell 1 — Sanity check (adata structure)
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse

GROUPBY = "leiden"   # 如果你用的是 leiden_bbknn_res1.0 就改这里
print(adata)
print("\n[obs columns]", list(adata.obs.columns))
print("[var columns]", list(adata.var.columns)[:20], "...")
print("\n[layers]", list(adata.layers.keys()))
print("[raw exists]", adata.raw is not None)

# X summary
X = adata.X
print("\n[X type]", type(X))
if sparse.issparse(X):
    print("[X shape]", X.shape, "nnz", X.nnz, "density", X.nnz/(X.shape[0]*X.shape[1]))
    print("[X min/max]", X.min(), X.max())
else:
    print("[X shape]", X.shape, "min/max", np.min(X), np.max(X))

# counts layer summary if exists
if "counts" in adata.layers:
    C = adata.layers["counts"]
    print("\n[counts layer type]", type(C))
    if sparse.issparse(C):
        print("[counts shape]", C.shape, "nnz", C.nnz)
        print("[counts min/max]", C.min(), C.max())
    else:
        print("[counts shape]", C.shape, "min/max", np.min(C), np.max(C))

# data layer summary if exists
if "data" in adata.layers:
    D = adata.layers["data"]
    print("\n[data layer type]", type(D))
    if sparse.issparse(D):
        print("[data shape]", D.shape, "nnz", D.nnz)
        print("[data min/max]", D.min(), D.max())
    else:
        print("[data shape]", D.shape, "min/max", np.min(D), np.max(D))


In [ ]:
# In[ ]: Cell 2 — Cluster/batch overview (counts table)
display(adata.obs[GROUPBY].value_counts().sort_index())

if "BATCH_KEY" in globals():
    print("\nBATCH_KEY =", BATCH_KEY)
    ct = pd.crosstab(adata.obs[GROUPBY], adata.obs[BATCH_KEY])
    display(ct)
    display((ct.T / ct.sum(1)).T.round(3))  # row-normalized
else:
    print("\n(BATCH_KEY not found in globals) - skip batch mixing table.")


In [ ]:
# In[ ]: Cell 3 — Compute dendrogram (required for dotplot dendrogram=True)
# This requires neighbors already computed. If you used BBKNN, you should already have neighbors.
# If dendrogram fails, set use_rep='X_pca' explicitly (assuming PCA exists).
try:
    sc.tl.dendrogram(adata, groupby=GROUPBY)
    print("✓ dendrogram computed:", f"dendrogram_{GROUPBY}" in adata.uns)
except Exception as e:
    print("dendrogram failed:", repr(e))
    if "X_pca" in adata.obsm_keys():
        print("Retry with use_rep='X_pca' ...")
        sc.tl.dendrogram(adata, groupby=GROUPBY, use_rep="X_pca")
        print("✓ dendrogram computed with X_pca")


In [ ]:
# In[ ]: Cell 4 — DotPlot (curated marker panel)
# 这组 marker 就是为了“看清 goblet/club/IFN/IEG/ductal/stress”结构
marker_panel = {
    "Mucous goblet": ["MUC5AC", "MUC5B", "MUC2", "MUC4", "MUC1", "MUC16", "BPIFB1", "PIGR", "AGR2", "SPDEF", "CLCA1", "TFF3"],
    "Club/secretory": ["SCGB1A1", "SCGB3A1", "SCGB3A2", "KRT19", "KRT8", "KRT18", "SLPI", "WFDC2", "MSMB", "VMO1"],
    "Inflammatory/antimicrobial": ["LCN2", "SAA1", "SAA2", "IL8", "CXCL1", "CXCL2", "DUOX2", "DUOXA2", "S100A8", "S100A9"],
    "IFN/ISG": ["ISG15", "IFI27", "IFITM1", "IFITM3", "MX1", "BST2", "OAS1", "OAS2", "STAT1"],
    "IEG/stress": ["FOS", "JUN", "ATF3", "DUSP1", "IER2", "HSPA1A", "HSPA1B"],
    "MHC-II": ["CD74", "HLA-DRA", "HLA-DRB1", "HLA-DPA1", "HLA-DPB1"],
    "Ductal/progenitor-like": ["SOX9", "PROM1", "MMP7", "EPCAM", "TACSTD2", "KRT7", "FOLR1", "PIP", "TCN1", "CLU"],
    "QC flags": ["MALAT1", "XIST", "HBB", "HBA1", "HBA2"]
}

# keep only genes that exist
genes_exist = set(adata.var_names)
marker_panel_filt = {k: [g for g in v if g in genes_exist] for k, v in marker_panel.items()}
missing = {k: [g for g in v if g not in genes_exist] for k, v in marker_panel.items()}

print("Missing genes (by panel):")
for k, v in missing.items():
    if len(v) > 0:
        print(f"  {k}: {v[:10]}" + (" ..." if len(v) > 10 else ""))

dp = sc.pl.dotplot(
    adata,
    var_names=marker_panel_filt,
    groupby=GROUPBY,
    dendrogram=True,
    standard_scale="var",      # 看相对表达更清楚
    expression_cutoff=0.0,
    mean_only_expressed=False,
    dot_max=0.6,
    swap_axes=True,
    show=True
)


In [ ]:
# In[ ]: Cell 5 — DotPlot (top markers per cluster from rank_genes_groups)
# 前提：你之前做过 sc.tl.rank_genes_groups(adata, groupby=GROUPBY, ...) 并保存在 adata.uns['rank_genes_groups']
# 这个图能帮我判断：你贴的 marker 是否是“稳定谱系”还是“状态/技术”
if "rank_genes_groups" not in adata.uns:
    print("rank_genes_groups not found. Run sc.tl.rank_genes_groups first.")
else:
    rg = adata.uns["rank_genes_groups"]
    groups = rg["names"].dtype.names
    top_n = 8

    top_genes = []
    for g in groups:
        top_genes.extend(list(rg["names"][g][:top_n]))
    # unique but keep order
    seen = set()
    top_genes = [x for x in top_genes if not (x in seen or seen.add(x))]
    top_genes = [g for g in top_genes if g in adata.var_names]

    print("Top genes used for dotplot:", len(top_genes))
    dp2 = sc.pl.dotplot(
        adata,
        var_names=top_genes,
        groupby=GROUPBY,
        dendrogram=True,
        standard_scale="var",
        swap_axes=True,
        show=True
    )


In [ ]:
# In[ ]: Cell 6 — Save dotplots (optional, consistent with your fig_dir settings)
from pathlib import Path

# reuse your existing settings if present
if "fig_dir" not in globals():
    fig_dir = Path("./figures_goblet")
fig_dir.mkdir(parents=True, exist_ok=True)

FIGURE_FORMAT = globals().get("FIGURE_FORMAT", "png")
FIGURE_DPI = globals().get("FIGURE_DPI", 300)

# Re-plot and save curated panel dotplot
plt.figure()
sc.pl.dotplot(
    adata,
    var_names=marker_panel_filt,
    groupby=GROUPBY,
    dendrogram=True,
    standard_scale="var",
    dot_max=0.6,
    swap_axes=True,
    show=False
)
plt.savefig(fig_dir / f"dotplot_marker_panel.{FIGURE_FORMAT}", dpi=FIGURE_DPI, bbox_inches="tight")
plt.show()
print(f"✓ Saved: dotplot_marker_panel.{FIGURE_FORMAT}")

# Re-plot and save top-markers dotplot (if available)
if "rank_genes_groups" in adata.uns:
    plt.figure()
    sc.pl.dotplot(
        adata,
        var_names=top_genes,
        groupby=GROUPBY,
        dendrogram=True,
        standard_scale="var",
        swap_axes=True,
        show=False
    )
    plt.savefig(fig_dir / f"dotplot_top_markers.{FIGURE_FORMAT}", dpi=FIGURE_DPI, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved: dotplot_top_markers.{FIGURE_FORMAT}")


In [ ]:
# In[ ]: Export dotplot gene panels to CSV (marker_panel + top_markers)
import pandas as pd
from pathlib import Path

# ---- output dir ----
if "fig_dir" not in globals():
    fig_dir = Path("./figures_goblet")
else:
    fig_dir = Path(fig_dir)
fig_dir.mkdir(parents=True, exist_ok=True)

GROUPBY = globals().get("GROUPBY", "leiden")

# ---- 1) curated marker panel CSV ----
# marker_panel_filt, missing should already exist from the dotplot cell
if "marker_panel_filt" not in globals():
    raise RuntimeError("marker_panel_filt not found. Run the marker dotplot cell first.")

rows = []
for panel, genes in marker_panel_filt.items():
    for g in genes:
        rows.append({"panel": panel, "gene": g})
df_marker_panel = pd.DataFrame(rows).sort_values(["panel", "gene"])
out1 = fig_dir / f"dotplot_marker_panel_genes_{GROUPBY}.csv"
df_marker_panel.to_csv(out1, index=False)

# also export missing genes for transparency
if "missing" in globals():
    miss_rows = []
    for panel, genes in missing.items():
        for g in genes:
            miss_rows.append({"panel": panel, "missing_gene": g})
    df_missing = pd.DataFrame(miss_rows).sort_values(["panel", "missing_gene"])
    out_miss = fig_dir / f"dotplot_marker_panel_missing_{GROUPBY}.csv"
    df_missing.to_csv(out_miss, index=False)
else:
    out_miss = None

print(f"✓ Saved curated marker genes: {out1}")
if out_miss is not None:
    print(f"✓ Saved missing genes list:  {out_miss}")

# ---- 2) top markers per cluster CSV ----
if "rank_genes_groups" not in adata.uns:
    print("rank_genes_groups not found in adata.uns — skip top-markers export.")
else:
    rg = adata.uns["rank_genes_groups"]
    groups = rg["names"].dtype.names
    top_n = 8  # match your dotplot cell; change if you used another

    long_rows = []
    for grp in groups:
        names = list(rg["names"][grp][:top_n])
        # include scores if present
        scores = list(rg["scores"][grp][:top_n]) if "scores" in rg else [None] * len(names)
        pvals_adj = list(rg["pvals_adj"][grp][:top_n]) if "pvals_adj" in rg else [None] * len(names)

        for i, (g, s, p) in enumerate(zip(names, scores, pvals_adj), start=1):
            long_rows.append({
                "cluster": grp,
                "rank": i,
                "gene": g,
                "score": s,
                "pvals_adj": p
            })

    df_top = pd.DataFrame(long_rows)
    out2 = fig_dir / f"dotplot_top_markers_{GROUPBY}_top{top_n}.csv"
    df_top.to_csv(out2, index=False)
    print(f"✓ Saved top markers:        {out2}")

    # optional: also export a wide table (each cluster as a column)
    wide = {grp: list(rg["names"][grp][:top_n]) for grp in groups}
    df_top_wide = pd.DataFrame(wide)
    out2w = fig_dir / f"dotplot_top_markers_{GROUPBY}_top{top_n}_wide.csv"
    df_top_wide.to_csv(out2w, index=False)
    print(f"✓ Saved top markers (wide): {out2w}")


In [ ]:
# In[ ]: Export DotPlot values (mean expression + percent expressed) to CSV
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse

# ========= USER SETTINGS (set these to match your dotplot) =========
GROUPBY = "leiden"                 # e.g. "leiden"
DOTPLOT_LAYER = None              # e.g. "counts" or "log1p" or None (use adata.X)
DOTPLOT_USE_RAW = False           # True if you used use_raw=True in dotplot
TOP_N = 8                         # top markers per cluster (match your dotplot)
OUTDIR = Path("./dotplot_csv")
OUTDIR.mkdir(parents=True, exist_ok=True)

# ========= helpers =========
def _get_X_and_varnames(adata, use_raw=False, layer=None):
    if use_raw:
        if adata.raw is None:
            raise ValueError("DOTPLOT_USE_RAW=True but adata.raw is None.")
        X = adata.raw.X
        var_names = adata.raw.var_names
    else:
        X = adata.layers[layer] if layer is not None else adata.X
        var_names = adata.var_names
    return X, var_names

def export_dotplot_values(adata, genes, groupby, out_prefix, use_raw=False, layer=None):
    X, var_names = _get_X_and_varnames(adata, use_raw=use_raw, layer=layer)

    # filter genes that exist
    genes = [g for g in genes if g in var_names]
    if len(genes) == 0:
        raise ValueError(f"No genes found in var_names for prefix={out_prefix}. Check gene symbols / use_raw / layer.")

    # index mapping
    gene_idx = pd.Index(var_names).get_indexer(genes)
    g = adata.obs[groupby].astype("category")
    groups = g.cat.categories.tolist()

    mean_mat = np.zeros((len(groups), len(genes)), dtype=np.float32)
    pct_mat  = np.zeros((len(groups), len(genes)), dtype=np.float32)

    # subset X to genes only once
    if sparse.issparse(X):
        Xg = X[:, gene_idx].tocsr()
    else:
        Xg = np.asarray(X)[:, gene_idx]

    for i, grp in enumerate(groups):
        idx = np.where(g.values == grp)[0]
        if idx.size == 0:
            continue

        if sparse.issparse(Xg):
            Xsub = Xg[idx, :]
            # mean expression
            mean_vec = np.asarray(Xsub.mean(axis=0)).ravel()
            # percent expressed (>0): treat non-zeros as 1 then mean
            Xbin = Xsub.copy()
            Xbin.data = np.ones_like(Xbin.data, dtype=np.float32)
            pct_vec = np.asarray(Xbin.mean(axis=0)).ravel()
        else:
            Xsub = Xg[idx, :]
            mean_vec = Xsub.mean(axis=0)
            pct_vec = (Xsub > 0).mean(axis=0)

        mean_mat[i, :] = mean_vec
        pct_mat[i, :] = pct_vec

    mean_df = pd.DataFrame(mean_mat, index=groups, columns=genes)
    pct_df  = pd.DataFrame(pct_mat * 100.0, index=groups, columns=genes)  # percent

    # wide CSV
    mean_path = OUTDIR / f"{out_prefix}.mean_expr_wide.csv"
    pct_path  = OUTDIR / f"{out_prefix}.pct_expr_wide.csv"
    mean_df.to_csv(mean_path, index=True)
    pct_df.to_csv(pct_path, index=True)

    # long CSV (easier to inspect / merge)
    long_df = (
        mean_df.stack().rename("mean_expr").reset_index()
        .rename(columns={"level_0": groupby, "level_1": "gene"})
        .merge(
            pct_df.stack().rename("pct_expr").reset_index()
              .rename(columns={"level_0": groupby, "level_1": "gene"}),
            on=[groupby, "gene"],
            how="left"
        )
    )
    long_path = OUTDIR / f"{out_prefix}.dotplot_values_long.csv"
    long_df.to_csv(long_path, index=False)

    print(f"✓ {out_prefix}")
    print(f"  - {mean_path}")
    print(f"  - {pct_path}")
    print(f"  - {long_path}")
    return mean_path, pct_path, long_path

# ========= 1) marker panel genes (your curated dotplot) =========
if "marker_panel_filt" not in globals():
    raise RuntimeError("marker_panel_filt not found. Run/build your marker panel (dict) first.")

marker_genes = []
for _, gs in marker_panel_filt.items():
    marker_genes.extend(list(gs))
marker_genes = list(dict.fromkeys(marker_genes))  # unique, keep order

export_dotplot_values(
    adata,
    genes=marker_genes,
    groupby=GROUPBY,
    out_prefix=f"dotplot_marker_panel_{GROUPBY}",
    use_raw=DOTPLOT_USE_RAW,
    layer=DOTPLOT_LAYER
)

# ========= 2) top markers genes (from rank_genes_groups) =========
if "rank_genes_groups" not in adata.uns:
    print("rank_genes_groups not found in adata.uns — skip top-markers export.")
else:
    rg = adata.uns["rank_genes_groups"]
    groups = rg["names"].dtype.names
    top_genes = []
    for grp in groups:
        top_genes.extend(list(rg["names"][grp][:TOP_N]))
    top_genes = list(dict.fromkeys(top_genes))

    export_dotplot_values(
        adata,
        genes=top_genes,
        groupby=GROUPBY,
        out_prefix=f"dotplot_top_markers_{GROUPBY}_top{TOP_N}",
        use_raw=DOTPLOT_USE_RAW,
        layer=DOTPLOT_LAYER
    )


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import sparse

GROUPBY = "leiden"
GENES = ["MUC5AC","MUC5B","MUC2","BPIFB1","PIGR","AGR2","CLCA1","TFF3","SCGB1A1","SCGB3A1","SCGB3A2",
         "SLPI","WFDC2","MSMB","VMO1","LCN2","SAA1","SAA2","IL8","CXCL1","CXCL2","DUOX2","DUOXA2",
         "S100A8","S100A9","IFI27","IFITM3","FOS","JUN","ATF3","DUSP1","IER2","HSPA1A","CD74",
         "HLA-DRA","HLA-DRB1","HLA-DPA1","MMP7","PIP","TCN1","CLU","MALAT1","XIST","HBB","HBA2"]

# 用未scale的表达层来算均值：优先 layers['log1p']；没有就用 adata.raw；再不行用 adata.X
LAYER_FOR_MEAN = None   # <- 你按实际改：如 "data" / None
USE_RAW_FOR_MEAN = False   # <- 若你把log1p放在adata.raw，就设True

OUTDIR = Path("/home/h2048/data/R/0105/goblet_bbknn/dotplot_csv_reexport")
OUTDIR.mkdir(exist_ok=True, parents=True)

def get_matrix(adata, use_raw=False, layer=None):
    if use_raw:
        if adata.raw is None:
            raise ValueError("USE_RAW_FOR_MEAN=True but adata.raw is None.")
        X = adata.raw.X
        var_names = adata.raw.var_names
    else:
        X = adata.layers[layer] if layer is not None else adata.X
        var_names = adata.var_names
    return X, var_names

X, var_names = get_matrix(adata, use_raw=USE_RAW_FOR_MEAN, layer=LAYER_FOR_MEAN)
genes = [g for g in GENES if g in var_names]
gi = pd.Index(var_names).get_indexer(genes)

g = adata.obs[GROUPBY].astype("category")
groups = g.cat.categories.tolist()

if sparse.issparse(X):
    Xg = X[:, gi].tocsr()
else:
    Xg = np.asarray(X)[:, gi]

mean_mat = np.zeros((len(groups), len(genes)), dtype=float)
pct_mat  = np.zeros((len(groups), len(genes)), dtype=float)

for i, grp in enumerate(groups):
    idx = np.where(g.values == grp)[0]
    if idx.size == 0:
        continue
    if sparse.issparse(Xg):
        Xsub = Xg[idx, :]
        mean_vec = np.asarray(Xsub.mean(axis=0)).ravel()
        Xbin = Xsub.copy()
        Xbin.data = np.ones_like(Xbin.data)
        pct_vec = np.asarray(Xbin.mean(axis=0)).ravel() * 100.0
    else:
        Xsub = Xg[idx, :]
        mean_vec = Xsub.mean(axis=0)
        pct_vec = (Xsub > 0).mean(axis=0) * 100.0
    mean_mat[i, :] = mean_vec
    pct_mat[i, :]  = pct_vec

mean_df = pd.DataFrame(mean_mat, index=groups, columns=genes)
pct_df  = pd.DataFrame(pct_mat,  index=groups, columns=genes)

mean_path = OUTDIR / f"{GROUPBY}.mean_log1p_wide.csv"
pct_path  = OUTDIR / f"{GROUPBY}.pct_expr_wide.csv"
mean_df.to_csv(mean_path)
pct_df.to_csv(pct_path)

print("Saved:")
print(mean_path)
print(pct_path)


In [ ]:
"""
================================================================================
Goblet Cell Marker Expression Analysis Pipeline
================================================================================
Purpose: Comprehensive marker gene analysis for goblet and secretory cells
Author: r2end
Date: 2025-01-06
Version: 1.0

Input: AnnData object with leiden clustering
Output: Dotplots, heatmaps, CSV tables, and analysis report

Key Features:
- Core goblet markers (MUC5AC, MUC5B, MUC2, TFF3, AGR2)
- Club/Secretory markers (SCGB1A1, SCGB3A1, BPIFA1, BPIFB1)
- Intestinal metaplasia detection (MUC2, FOXA1, KLF4)
- Inflammatory activation markers (SAA1/2, CXCL1, CSF3)
- Automated cell type annotation suggestions
================================================================================
"""

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime

# ===== Configuration =====
print("="*80)
print("GOBLET CELL MARKER ANALYSIS PIPELINE")
print("="*80)
print(f"Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")



# Analysis parameters
CLUSTER_KEY = 'leiden'  # Clustering column name
USE_RAW = True  # Use .raw for gene expression
N_JOBS = 8  # Number of parallel jobs

# Key clusters to focus on (adjust based on your data)
KEY_CLUSTERS = ['10', '11', '12', '13', '14', '15', '22']

print(f"\n{'='*80}")
print("Configuration Summary:")
print(f"{'='*80}")
print(f"Input file: {INPUT_H5AD}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Cluster key: {CLUSTER_KEY}")
print(f"Use raw data: {USE_RAW}")
print(f"Key clusters: {', '.join(KEY_CLUSTERS)}")
print(f"{'='*80}\n")

# ===== 1. Load Data =====
print("="*80)
print("STEP 1: Loading Data")
print("="*80)

try:
    adata = sc.read_h5ad(INPUT_H5AD)
    print(f"✓ Successfully loaded: {INPUT_H5AD}")
    print(f"  Shape: {adata.shape} (cells × genes)")
    print(f"  Clusters: {adata.obs[CLUSTER_KEY].nunique()}")
    print(f"  Has .raw: {adata.raw is not None}")
    
    # Check cluster distribution
    print("\nCluster distribution:")
    cluster_counts = adata.obs[CLUSTER_KEY].value_counts().sort_index()
    for cluster, count in cluster_counts.items():
        pct = count / adata.n_obs * 100
        print(f"  Cluster {cluster}: {count:,} cells ({pct:.2f}%)")
    
except Exception as e:
    print(f"✗ Error loading data: {e}")
    raise

# ===== 2. Define Goblet Cell Marker Genes =====
print("\n" + "="*80)
print("STEP 2: Defining Marker Genes")
print("="*80)

goblet_markers = {
    # === Core Goblet Markers (核心杯状细胞标志) ===
    'Core_Goblet': [
        'MUC5AC',    # Respiratory goblet cell gold standard
        'MUC5B',     # Airway mucus
        'MUC2',      # Intestinal goblet cell (pathological)
        'MUC4',      # Membrane-bound mucin
        'MUC16',     # Membrane-bound mucin
        'TFF3',      # Trefoil factor, mucosal repair
        'AGR2',      # Anterior gradient 2, mucin folding
        'SPDEF',     # SAM pointed domain ETS factor
    ],
    
    # === Secretory/Club Cell Markers (分泌细胞标志) ===
    'Club_Secretory': [
        'SCGB1A1',   # Club cell secretory protein
        'SCGB3A1',   # Secretoglobin 3A1
        'CYP2F1',    # Cytochrome P450 (detoxification)
        'CYP2J2',    # Cytochrome P450
        'BPIFA1',    # BPI fold containing family A
        'BPIFB1',    # BPI fold containing family B
    ],
    
    # === Mucin Secretion & Processing (黏液分泌与加工) ===
    'Mucin_Processing': [
        'PIGR',      # Polymeric Ig receptor (IgA transport)
        'LCN2',      # Lipocalin 2 (inflammation-related)
        'TFF1',      # Trefoil factor 1
        'FCGBP',     # Fc fragment binding protein
        'CLCA1',     # Chloride channel accessory 1
        'CLCA2',     # Chloride channel accessory 2
        'CLCA4',     # Chloride channel accessory 4
    ],
    
    # === Intestinal Metaplasia Markers (肠化生标志) ===
    'Intestinal_Metaplasia': [
        'MUC2',      # Intestinal goblet cell-specific
        'FOXA1',     # Forkhead box A1 (intestinal TF)
        'FOXA2',     # Forkhead box A2
        'CDX2',      # Caudal type homeobox 2
        'KLF4',      # Kruppel-like factor 4
        'ATOH1',     # Atonal homolog 1
    ],
    
    # === Inflammatory/Activated State (炎症/激活状态) ===
    'Inflammatory': [
        'SAA1',      # Serum amyloid A1
        'SAA2',      # Serum amyloid A2
        'SLURP2',    # Secreted Ly6/PLAUR domain containing 2
        'C15orf48',  # Chromosome 15 open reading frame 48
        'CXCL1',     # C-X-C motif chemokine ligand 1
        'CXCL6',     # C-X-C motif chemokine ligand 6
        'IL8',       # Interleukin 8 (CXCL8)
        'CSF3',      # Colony stimulating factor 3 (G-CSF)
    ],
    
    # === Goblet Cell Maturation (杯状细胞成熟) ===
    'Maturation': [
        'XBP1',      # X-box binding protein 1 (ER stress)
        'ELF3',      # E74 like ETS TF 3
        'WFDC2',     # WAP four-disulfide core domain 2
        'SLPI',      # Secretory leukocyte peptidase inhibitor
        'MSMB',      # Microseminoprotein beta
    ],
    
    # === Metabolism & Stress Response (代谢与应激) ===
    'Metabolism': [
        'GPX2',      # Glutathione peroxidase 2
        'PRDX1',     # Peroxiredoxin 1
        'ALDH3A1',   # Aldehyde dehydrogenase 3A1
        'SELENOP',   # Selenoprotein P
        'MT1X',      # Metallothionein 1X
        'MT2A',      # Metallothionein 2A
    ],
}

# Merge all markers and remove duplicates
all_markers = []
for category, genes in goblet_markers.items():
    all_markers.extend(genes)
all_markers = list(set(all_markers))

# Check which genes are available in the dataset
available_markers = [g for g in all_markers if g in adata.var_names]
missing_markers = [g for g in all_markers if g not in adata.var_names]

print(f"\n{'='*80}")
print("Marker Gene Availability:")
print(f"{'='*80}")
print(f"Total markers defined: {len(all_markers)}")
print(f"Available in dataset: {len(available_markers)} ({len(available_markers)/len(all_markers)*100:.1f}%)")
print(f"Missing markers: {len(missing_markers)}")

if missing_markers:
    print(f"\nMissing genes: {', '.join(missing_markers)}")

# Check availability by category
available_by_category = {}
print(f"\n{'='*80}")
print("Availability by Category:")
print(f"{'='*80}")

for category, genes in goblet_markers.items():
    available = [g for g in genes if g in adata.var_names]
    available_by_category[category] = available
    pct = len(available) / len(genes) * 100 if genes else 0
    print(f"\n{category}: {len(available)}/{len(genes)} ({pct:.1f}%)")
    print(f"  Available: {', '.join(available) if available else 'None'}")
    
    if len(available) < len(genes):
        missing = [g for g in genes if g not in adata.var_names]
        print(f"  Missing: {', '.join(missing)}")

# ===== 3. Generate Dotplots =====
print("\n" + "="*80)
print("STEP 3: Generating Dotplots")
print("="*80)

# 3.1 Core goblet markers
if available_by_category['Core_Goblet']:
    print("\n[3.1] Creating core goblet markers dotplot...")
    try:
        # Determine cluster order (focus on key clusters first)
        all_clusters = sorted(adata.obs[CLUSTER_KEY].unique().astype(str), key=int)
        ordered_clusters = [c for c in KEY_CLUSTERS if c in all_clusters]
        ordered_clusters += [c for c in all_clusters if c not in ordered_clusters]
        
        fig, ax = plt.subplots(figsize=(10, 6))
        sc.pl.dotplot(
            adata,
            var_names=available_by_category['Core_Goblet'],
            groupby=CLUSTER_KEY,
            dendrogram=False,
            categories_order=ordered_clusters[:15],  # Top 15 clusters
            use_raw=USE_RAW,
            standard_scale='var',
            cmap='Reds',
            ax=ax,
            show=False
        )
        plt.title('Core Goblet Cell Markers', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f"{FIGURE_DIR}/dotplot_core_goblet_markers.pdf", dpi=300, bbox_inches='tight')
        plt.savefig(f"{FIGURE_DIR}/dotplot_core_goblet_markers.png", dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: dotplot_core_goblet_markers")
    except Exception as e:
        print(f"  ✗ Error: {e}")

# 3.2 Club/Secretory vs Goblet transition
if available_by_category['Club_Secretory'] and available_by_category['Core_Goblet']:
    print("\n[3.2] Creating Club → Goblet transition dotplot...")
    try:
        combined_secretory = (available_by_category['Club_Secretory'] + 
                            available_by_category['Core_Goblet'][:4])
        
        fig, ax = plt.subplots(figsize=(12, 6))
        sc.pl.dotplot(
            adata,
            var_names=combined_secretory,
            groupby=CLUSTER_KEY,
            dendrogram=False,
            categories_order=ordered_clusters[:15],
            use_raw=USE_RAW,
            standard_scale='var',
            cmap='RdYlBu_r',
            ax=ax,
            show=False
        )
        plt.title('Club → Goblet Transition Markers', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f"{FIGURE_DIR}/dotplot_club_to_goblet.pdf", dpi=300, bbox_inches='tight')
        plt.savefig(f"{FIGURE_DIR}/dotplot_club_to_goblet.png", dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: dotplot_club_to_goblet")
    except Exception as e:
        print(f"  ✗ Error: {e}")

# 3.3 Pathological markers (metaplasia + inflammation)
print("\n[3.3] Creating pathological markers dotplot...")
try:
    pathology_markers = (
        available_by_category['Intestinal_Metaplasia'][:4] +
        available_by_category['Inflammatory'][:6]
    )
    
    if pathology_markers:
        fig, ax = plt.subplots(figsize=(10, 8))
        sc.pl.dotplot(
            adata,
            var_names=pathology_markers,
            groupby=CLUSTER_KEY,
            dendrogram=False,
            use_raw=USE_RAW,
            standard_scale='var',
            cmap='OrRd',
            ax=ax,
            show=False
        )
        plt.title('Pathological Markers (Metaplasia & Inflammation)', 
                 fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f"{FIGURE_DIR}/dotplot_pathology_markers.pdf", dpi=300, bbox_inches='tight')
        plt.savefig(f"{FIGURE_DIR}/dotplot_pathology_markers.png", dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: dotplot_pathology_markers")
    else:
        print("  ⚠ No pathological markers available")
except Exception as e:
    print(f"  ✗ Error: {e}")

# 3.4 Comprehensive dotplot
print("\n[3.4] Creating comprehensive dotplot...")
try:
    representative_markers = []
    for category, genes in available_by_category.items():
        representative_markers.extend(genes[:3])  # Top 3 per category
    representative_markers = list(set(representative_markers))
    
    if representative_markers:
        fig = plt.figure(figsize=(14, 10))
        sc.pl.dotplot(
            adata,
            var_names=representative_markers,
            groupby=CLUSTER_KEY,
            dendrogram=True,
            use_raw=USE_RAW,
            standard_scale='var',
            cmap='viridis',
            show=False
        )
        plt.title('Comprehensive Goblet-Related Markers (All Clusters)', 
                 fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f"{FIGURE_DIR}/dotplot_comprehensive.pdf", dpi=300, bbox_inches='tight')
        plt.savefig(f"{FIGURE_DIR}/dotplot_comprehensive.png", dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: dotplot_comprehensive")
except Exception as e:
    print(f"  ✗ Error: {e}")

# ===== 4. Extract and Export Expression Data =====
print("\n" + "="*80)
print("STEP 4: Extracting Expression Data")
print("="*80)

print("\n[4.1] Computing cluster-wise expression statistics...")

mean_expr_df = pd.DataFrame()
pct_expr_df = pd.DataFrame()

for cluster in sorted(adata.obs[CLUSTER_KEY].unique(), key=lambda x: int(str(x))):
    cluster_cells = adata.obs[CLUSTER_KEY] == cluster
    n_cells = cluster_cells.sum()
    
    print(f"  Processing Cluster {cluster}: {n_cells} cells")
    
    # Get expression data
    if adata.raw is not None and USE_RAW:
        cluster_data = adata.raw[cluster_cells, available_markers].X
    else:
        cluster_data = adata[cluster_cells, available_markers].X
    
    # Convert to dense array if sparse
    if hasattr(cluster_data, 'toarray'):
        cluster_data = cluster_data.toarray()
    
    # Calculate statistics
    mean_expr = np.mean(cluster_data, axis=0)
    mean_expr_df[f'Cluster_{cluster}'] = mean_expr
    
    pct_expr = (cluster_data > 0).sum(axis=0) / cluster_data.shape[0] * 100
    pct_expr_df[f'Cluster_{cluster}'] = pct_expr

mean_expr_df.index = available_markers
pct_expr_df.index = available_markers

# Add gene category information
gene_category = {}
for category, genes in available_by_category.items():
    for gene in genes:
        if gene in available_markers:
            gene_category[gene] = category

mean_expr_df['Gene_Category'] = mean_expr_df.index.map(gene_category)
pct_expr_df['Gene_Category'] = pct_expr_df.index.map(gene_category)

# Save tables
print("\n[4.2] Saving expression tables...")
mean_expr_df.to_csv(f"{TABLE_DIR}/goblet_markers_mean_expression.csv")
print(f"  ✓ Saved: goblet_markers_mean_expression.csv")

pct_expr_df.to_csv(f"{TABLE_DIR}/goblet_markers_pct_cells.csv")
print(f"  ✓ Saved: goblet_markers_pct_cells.csv")

# Combined table
combined_df = pd.DataFrame()
for cluster in sorted(adata.obs[CLUSTER_KEY].unique(), key=lambda x: int(str(x))):
    cluster_label = f'Cluster_{cluster}'
    if cluster_label in mean_expr_df.columns:
        combined_df[f'{cluster_label}_mean'] = mean_expr_df[cluster_label]
        combined_df[f'{cluster_label}_pct'] = pct_expr_df[cluster_label]

combined_df['Gene_Category'] = mean_expr_df['Gene_Category']
combined_df.to_csv(f"{TABLE_DIR}/goblet_markers_combined.csv")
print(f"  ✓ Saved: goblet_markers_combined.csv")

# Key clusters detailed table
key_clusters_df = pd.DataFrame()
for cluster in KEY_CLUSTERS:
    cluster_label = f'Cluster_{cluster}'
    if cluster_label in mean_expr_df.columns:
        key_clusters_df[f'C{cluster}_mean'] = mean_expr_df[cluster_label]
        key_clusters_df[f'C{cluster}_pct'] = pct_expr_df[cluster_label]

if not key_clusters_df.empty:
    key_clusters_df['Gene_Category'] = mean_expr_df['Gene_Category']
    key_clusters_df = key_clusters_df.sort_values('Gene_Category')
    key_clusters_df.to_csv(f"{TABLE_DIR}/goblet_key_clusters_detailed.csv")
    print(f"  ✓ Saved: goblet_key_clusters_detailed.csv")

# ===== 5. Generate Heatmaps =====
print("\n" + "="*80)
print("STEP 5: Generating Heatmaps")
print("="*80)

# 5.1 Core goblet markers heatmap
if available_by_category['Core_Goblet']:
    print("\n[5.1] Creating core goblet markers heatmap...")
    try:
        core_genes = [g for g in available_by_category['Core_Goblet'] if g in available_markers]
        
        if core_genes:
            # Select clusters for heatmap
            cluster_cols = [f'Cluster_{c}' for c in KEY_CLUSTERS 
                          if f'Cluster_{c}' in mean_expr_df.columns]
            
            if cluster_cols:
                heatmap_data = mean_expr_df.loc[core_genes, cluster_cols].T
                
                plt.figure(figsize=(10, 6))
                sns.heatmap(
                    heatmap_data,
                    cmap='RdYlBu_r',
                    center=heatmap_data.values.mean(),
                    cbar_kws={'label': 'Mean Expression (log1p)'},
                    yticklabels=True,
                    xticklabels=True,
                    linewidths=0.5
                )
                plt.title('Core Goblet Markers - Mean Expression', 
                         fontsize=14, fontweight='bold')
                plt.xlabel('Gene', fontsize=12)
                plt.ylabel('Cluster', fontsize=12)
                plt.tight_layout()
                plt.savefig(f"{FIGURE_DIR}/heatmap_core_goblet.pdf", dpi=300, bbox_inches='tight')
                plt.savefig(f"{FIGURE_DIR}/heatmap_core_goblet.png", dpi=300, bbox_inches='tight')
                plt.close()
                print("  ✓ Saved: heatmap_core_goblet")
    except Exception as e:
        print(f"  ✗ Error: {e}")

# 5.2 All markers for key clusters
print("\n[5.2] Creating comprehensive markers heatmap...")
try:
    cluster_cols = [f'Cluster_{c}' for c in KEY_CLUSTERS 
                   if f'Cluster_{c}' in mean_expr_df.columns]
    
    if cluster_cols and available_markers:
        heatmap_data = mean_expr_df.loc[available_markers, cluster_cols].T
        
        plt.figure(figsize=(16, 8))
        sns.heatmap(
            heatmap_data,
            cmap='viridis',
            cbar_kws={'label': 'Mean Expression (log1p)'},
            yticklabels=True,
            xticklabels=True,
            linewidths=0.5
        )
        plt.title('Key Goblet/Secretory Clusters - All Markers', 
                 fontsize=14, fontweight='bold')
        plt.xlabel('Gene', fontsize=12)
        plt.ylabel('Cluster', fontsize=12)
        plt.tight_layout()
        plt.savefig(f"{FIGURE_DIR}/heatmap_key_clusters_all.pdf", dpi=300, bbox_inches='tight')
        plt.savefig(f"{FIGURE_DIR}/heatmap_key_clusters_all.png", dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: heatmap_key_clusters_all")
except Exception as e:
    print(f"  ✗ Error: {e}")

# ===== 6. Cluster Characterization Summary =====
print("\n" + "="*80)
print("STEP 6: Cluster Characterization")
print("="*80)

cluster_summary = []

for cluster in KEY_CLUSTERS:
    cluster_label = f'Cluster_{cluster}'
    if cluster_label not in mean_expr_df.columns:
        continue
    
    n_cells = (adata.obs[CLUSTER_KEY] == cluster).sum()
    pct_total = n_cells / adata.n_obs * 100
    
    cluster_info = {
        'Cluster': cluster,
        'N_cells': n_cells,
        'Pct_total': pct_total
    }
    
    # Calculate category-wise statistics
    for category, genes in available_by_category.items():
        available_genes = [g for g in genes if g in available_markers]
        
        if available_genes:
            mean_values = mean_expr_df.loc[available_genes, cluster_label]
            pct_values = pct_expr_df.loc[available_genes, cluster_label]
            
            cluster_info[f'{category}_mean_expr'] = mean_values.mean()
            cluster_info[f'{category}_mean_pct'] = pct_values.mean()
            cluster_info[f'{category}_n_expressed'] = (mean_values > 0.5).sum()
    
    cluster_summary.append(cluster_info)

summary_df = pd.DataFrame(cluster_summary)
summary_df.to_csv(f"{TABLE_DIR}/cluster_characterization_summary.csv", index=False)
print(f"✓ Saved: cluster_characterization_summary.csv")

print("\nCluster Summary:")
print(summary_df[['Cluster', 'N_cells', 'Pct_total']].to_string(index=False))

# ===== 7. Specific Comparisons =====
print("\n" + "="*80)
print("STEP 7: Generating Comparison Plots")
print("="*80)

# 7.1 MUC5AC vs MUC2 vs MUC5B scatter plots
muc_genes = [g for g in ['MUC5AC', 'MUC2', 'MUC5B', 'SCGB1A1'] if g in available_markers]

if len(muc_genes) >= 2:
    print("\n[7.1] Creating mucin comparison scatter plots...")
    try:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        
        # Plot 1: MUC5AC vs MUC2 (intestinal metaplasia)
        if 'MUC5AC' in muc_genes and 'MUC2' in muc_genes:
            ax = axes[0]
            for cluster in KEY_CLUSTERS:
                cluster_label = f'Cluster_{cluster}'
                if cluster_label in mean_expr_df.columns:
                    x_val = mean_expr_df.loc['MUC5AC', cluster_label]
                    y_val = mean_expr_df.loc['MUC2', cluster_label]
                    
                    ax.scatter(x_val, y_val, s=200, alpha=0.7, label=f'C{cluster}')
                    ax.text(x_val, y_val, cluster, fontsize=10, ha='center', va='center')
            
            ax.set_xlabel('MUC5AC Expression', fontsize=12, fontweight='bold')
            ax.set_ylabel('MUC2 Expression', fontsize=12, fontweight='bold')
            ax.set_title('MUC5AC vs MUC2 (Intestinal Metaplasia)', 
                        fontsize=12, fontweight='bold')
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
            ax.grid(alpha=0.3)
        
        # Plot 2: SCGB1A1 vs MUC5AC (Club→Goblet transition)
        if 'SCGB1A1' in muc_genes and 'MUC5AC' in muc_genes:
            ax = axes[1]
            for cluster in KEY_CLUSTERS:
                cluster_label = f'Cluster_{cluster}'
                if cluster_label in mean_expr_df.columns:
                    x_val = mean_expr_df.loc['SCGB1A1', cluster_label]
                    y_val = mean_expr_df.loc['MUC5AC', cluster_label]
                    
                    ax.scatter(x_val, y_val, s=200, alpha=0.7, label=f'C{cluster}')
                    ax.text(x_val, y_val, cluster, fontsize=10, ha='center', va='center')
            
            ax.set_xlabel('SCGB1A1 Expression (Club)', fontsize=12, fontweight='bold')
            ax.set_ylabel('MUC5AC Expression (Goblet)', fontsize=12, fontweight='bold')
            ax.set_title('Club vs Goblet Transition', fontsize=12, fontweight='bold')
            ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
            ax.grid(alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f"{FIGURE_DIR}/scatter_mucin_comparison.pdf", dpi=300, bbox_inches='tight')
        plt.savefig(f"{FIGURE_DIR}/scatter_mucin_comparison.png", dpi=300, bbox_inches='tight')
        plt.close()
        print("  ✓ Saved: scatter_mucin_comparison")
    except Exception as e:
        print(f"  ✗ Error: {e}")

# ===== 8. Generate Annotation Recommendations =====
print("\n" + "="*80)
print("STEP 8: Generating Annotation Recommendations")
print("="*80)

annotation_dict = {}

for cluster in KEY_CLUSTERS:
    cluster_label = f'Cluster_{cluster}'
    if cluster_label not in mean_expr_df.columns:
        continue
    
    # Get expression characteristics
    muc5ac = mean_expr_df.loc['MUC5AC', cluster_label] if 'MUC5AC' in available_markers else 0
    muc2 = mean_expr_df.loc['MUC2', cluster_label] if 'MUC2' in available_markers else 0
    muc5b = mean_expr_df.loc['MUC5B', cluster_label] if 'MUC5B' in available_markers else 0
    scgb1a1 = mean_expr_df.loc['SCGB1A1', cluster_label] if 'SCGB1A1' in available_markers else 0
    
    # Calculate category scores
    core_goblet = mean_expr_df.loc[available_by_category['Core_Goblet'], cluster_label].mean() if available_by_category['Core_Goblet'] else 0
    club_expr = mean_expr_df.loc[available_by_category['Club_Secretory'], cluster_label].mean() if available_by_category['Club_Secretory'] else 0
    
    # Decision tree for cell type annotation
    if scgb1a1 > 2.0 and muc5ac < 1.0:
        cell_type = "Club_cells"
        confidence = "High"
    elif scgb1a1 > 2.0 and muc5ac > 1.0:
        cell_type = "Club_Goblet_transition"
        confidence = "High"
    elif muc5ac > 2.0 and muc2 > 2.0:
        cell_type = "Goblet_hyperplastic_MUC2+"
        confidence = "High"
    elif muc5ac > 2.0:
        cell_type = "Goblet_mature"
        confidence = "High"
    elif muc5ac > 1.0:
        cell_type = "Goblet_early"
        confidence = "Medium"
    elif muc5b > 2.0:
        cell_type = "Mucous_secretory"
        confidence = "Medium"
    else:
        cell_type = "Other_epithelial"
        confidence = "Low"
    
    annotation_dict[cluster] = {
        'Cell_Type': cell_type,
        'Confidence': confidence,
        'MUC5AC': f"{muc5ac:.2f}",
        'MUC2': f"{muc2:.2f}",
        'MUC5B': f"{muc5b:.2f}",
        'SCGB1A1': f"{scgb1a1:.2f}",
        'Core_Goblet_Score': f"{core_goblet:.2f}",
        'Club_Score': f"{club_expr:.2f}"
    }

annotation_df = pd.DataFrame.from_dict(annotation_dict, orient='index')
annotation_df.index.name = 'Cluster'
annotation_df.to_csv(f"{TABLE_DIR}/recommended_annotations.csv")
print(f"✓ Saved: recommended_annotations.csv\n")

print("Recommended Annotations:")
print(annotation_df[['Cell_Type', 'Confidence']].to_string())

# ===== 9. Generate Final Report =====
print("\n" + "="*80)
print("STEP 9: Generating Final Report")
print("="*80)

report_lines = [
    "=" * 80,
    "GOBLET CELL MARKER EXPRESSION ANALYSIS REPORT",
    "=" * 80,
    f"\nGenerated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    f"Input file: {INPUT_H5AD}",
    f"Output directory: {OUTPUT_DIR}",
    f"\nDataset Summary:",
    f"  Total cells: {adata.n_obs:,}",
    f"  Total genes: {adata.n_vars:,}",
    f"  Number of clusters: {adata.obs[CLUSTER_KEY].nunique()}",
    f"  Has .raw: {adata.raw is not None}",
    "\n" + "=" * 80,
    "MARKER GENE SUMMARY",
    "=" * 80,
    f"\nTotal markers defined: {len(all_markers)}",
    f"Available in dataset: {len(available_markers)} ({len(available_markers)/len(all_markers)*100:.1f}%)",
]

for category, genes in available_by_category.items():
    pct = len(genes) / len(goblet_markers[category]) * 100 if goblet_markers[category] else 0
    report_lines.append(f"\n{category}: {len(genes)}/{len(goblet_markers[category])} ({pct:.1f}%)")
    report_lines.append(f"  Genes: {', '.join(genes) if genes else 'None'}")

report_lines.extend([
    "\n" + "=" * 80,
    "KEY CLUSTER CHARACTERIZATION",
    "=" * 80,
])

for _, row in summary_df.iterrows():
    cluster = row['Cluster']
    report_lines.append(f"\nCluster {cluster}:")
    report_lines.append(f"  Cells: {row['N_cells']:,} ({row['Pct_total']:.2f}%)")
    report_lines.append(f"  Core Goblet Score: {row.get('Core_Goblet_mean_expr', 0):.3f}")
    report_lines.append(f"  Club/Secretory Score: {row.get('Club_Secretory_mean_expr', 0):.3f}")
    report_lines.append(f"  Inflammatory Score: {row.get('Inflammatory_mean_expr', 0):.3f}")

report_lines.extend([
    "\n" + "=" * 80,
    "PATHOLOGICAL FEATURES",
    "=" * 80,
])

# Check for intestinal metaplasia (MUC2)
if 'MUC2' in available_markers:
    muc2_high = []
    for cluster in KEY_CLUSTERS:
        cluster_label = f'Cluster_{cluster}'
        if cluster_label in mean_expr_df.columns:
            muc2_expr = mean_expr_df.loc['MUC2', cluster_label]
            muc2_pct = pct_expr_df.loc['MUC2', cluster_label]
            if muc2_expr > 1.0 and muc2_pct > 20:
                muc2_high.append(f"Cluster {cluster} (expr={muc2_expr:.2f}, pct={muc2_pct:.1f}%)")
    
    if muc2_high:
        report_lines.append("\n⚠️  INTESTINAL METAPLASIA DETECTED (MUC2+ clusters):")
        for info in muc2_high:
            report_lines.append(f"  - {info}")
    else:
        report_lines.append("\n✓ No significant MUC2 expression detected")

# Check for inflammatory activation (SAA1/2)
saa_genes = [g for g in ['SAA1', 'SAA2'] if g in available_markers]
if saa_genes:
    inflammatory = []
    for cluster in KEY_CLUSTERS:
        cluster_label = f'Cluster_{cluster}'
        if cluster_label in mean_expr_df.columns:
            saa_expr = mean_expr_df.loc[saa_genes, cluster_label].mean()
            if saa_expr > 1.0:
                inflammatory.append(f"Cluster {cluster} (SAA expr={saa_expr:.2f})")
    
    if inflammatory:
        report_lines.append("\n⚠️  INFLAMMATORY ACTIVATION DETECTED (SAA1/2+ clusters):")
        for info in inflammatory:
            report_lines.append(f"  - {info}")

report_lines.extend([
    "\n" + "=" * 80,
    "RECOMMENDED ANNOTATIONS",
    "=" * 80,
])

for cluster, info in annotation_dict.items():
    report_lines.append(f"\nCluster {cluster}: {info['Cell_Type']} (Confidence: {info['Confidence']})")
    report_lines.append(f"  MUC5AC={info['MUC5AC']}, MUC2={info['MUC2']}, SCGB1A1={info['SCGB1A1']}")

report_lines.extend([
    "\n" + "=" * 80,
    "OUTPUT FILES",
    "=" * 80,
    f"\nFigures: {FIGURE_DIR}",
    "  - dotplot_core_goblet_markers.pdf/png",
    "  - dotplot_club_to_goblet.pdf/png",
    "  - dotplot_pathology_markers.pdf/png",
    "  - dotplot_comprehensive.pdf/png",
    "  - heatmap_core_goblet.pdf/png",
    "  - heatmap_key_clusters_all.pdf/png",
    "  - scatter_mucin_comparison.pdf/png",
    f"\nTables: {TABLE_DIR}",
    "  - goblet_markers_mean_expression.csv",
    "  - goblet_markers_pct_cells.csv",
    "  - goblet_markers_combined.csv",
    "  - goblet_key_clusters_detailed.csv",
    "  - cluster_characterization_summary.csv",
    "  - recommended_annotations.csv",
    "  - analysis_report.txt",
    "\n" + "=" * 80,
    "END OF REPORT",
    "=" * 80,
])

# Save report
report_text = "\n".join(report_lines)
with open(f"{OUTPUT_DIR}/analysis_report.txt", 'w') as f:
    f.write(report_text)

print(f"\n✓ Report saved to: {OUTPUT_DIR}/analysis_report.txt")

# Print report to console
print("\n" + report_text)

# ===== 10. Summary and Next Steps =====
print("\n" + "="*80)
print("🎉 ANALYSIS COMPLETE! 🎉")
print("="*80)
print(f"\nExecution time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"All outputs saved to: {OUTPUT_DIR}")

print("\n📊 Generated Files Summary:")
print(f"  Figures: {len([f for f in os.listdir(FIGURE_DIR) if f.endswith(('.pdf', '.png'))])} files")
print(f"  Tables: {len([f for f in os.listdir(TABLE_DIR) if f.endswith('.csv')])} files")

print("\n📝 Next Steps:")
print("1. Review dotplots to visualize marker expression patterns")
print("2. Check CSV files for detailed quantitative data")
print("3. Validate MUC2+ clusters with IHC/IF if intestinal metaplasia detected")
print("4. Apply recommended annotations or adjust based on biological knowledge")
print("5. Consider trajectory analysis for Club→Goblet differentiation")
print("6. Investigate inflammatory clusters (SAA1/2+) in disease context")

print("\n💡 Key Findings:")
if 'MUC2' in available_markers:
    muc2_clusters = [c for c in KEY_CLUSTERS 
                     if f'Cluster_{c}' in mean_expr_df.columns 
                     and mean_expr_df.loc['MUC2', f'Cluster_{c}'] > 1.0]
    if muc2_clusters:
        print(f"  ⚠️  Intestinal metaplasia detected in clusters: {', '.join(muc2_clusters)}")
    else:
        print(f"  ✓ No significant intestinal metaplasia")

if saa_genes:
    saa_clusters = [c for c in KEY_CLUSTERS 
                    if f'Cluster_{c}' in mean_expr_df.columns 
                    and mean_expr_df.loc[saa_genes, f'Cluster_{c}'].mean() > 1.0]
    if saa_clusters:
        print(f"  ⚠️  Inflammatory activation in clusters: {', '.join(saa_clusters)}")

print("\n" + "="*80)
print("Analysis pipeline completed successfully!")
print("="*80)

In [ ]:
import pandas as pd

# ===== Goblet Cell Complete Annotation System (REVISED) =====
# Level 2 now represents shared functional states

annotation_data = {
    'cluster': [
        '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',
        '10', '11', '12', '13', '14', '15', '16', '17', '18', '19',
        '20', '21', '22', '23', '24', '25'
    ],
    
    # ===== Level 1: Major Cell Lineage =====
    'cell_type_level_2': [
        'Goblet',           # 0
        'Goblet',            # 1
        'Goblet',           # 2
        'Basal',            # 3
        'Goblet',           # 4
        'Basal',            # 5
        'Secretory',        # 6
        'Goblet',           # 7
        'Goblet',           # 8
        'Goblet',           # 9
        'Goblet',           # 10
        'Goblet',           # 11
        'Secretory',        # 12
        'Secretory',        # 13
        'Goblet',           # 14
        'Secretory',        # 15
        'Goblet',           # 16
        'Goblet',           # 17
        'Goblet',           # 18
        'Basal',            # 19
        'Basal',            # 20
        'Goblet',           # 21
        'Secretory',        # 22
        'Secretory',             # 23
        'Secretory',            # 24
        'Goblet',           # 25
    ],
    
    # ===== Level 2: Functional State (MERGED) =====
    'cell_type_level_3': [
        'Proliferating',    # 0 - high metabolic, cycling
        'Differentiating',  # 1 - basal to secretory transition
        'Mature',           # 2 - immune active but mature
        'Quiescent',        # 3 - low expression, resting
        'Mature',           # 4 - mature high activity
        'Proliferating',    # 5 - cycling basal
        'Mature',           # 6 - mature club
        'Mature',           # 7 - well differentiated
        'Mature',           # 8 - moderate mature
        'Proliferating',    # 9 - metabolic active
        'Mature',           # 10 - classic mature
        'Hyperplastic',     # 11 - pathological hyperplasia
        'Transitioning',    # 12 - club-goblet hybrid
        'Transitioning',    # 13 - early transition
        'Mature',           # 14 - mature secretory
        'Proliferating',    # 15 - proliferating secretory
        'Inflammatory',     # 16 - IFN/SAA activated
        'Mature',           # 17 - moderate mature
        'Differentiating',  # 18 - early maturing
        'Quiescent',        # 19 - very low expression
        'Proliferating',    # 20 - cycling basal
        'Stressed',         # 21 - ER stress, inflammatory
        'Transitioning',    # 22 - inflammation-driven transition
        'Specialized',      # 23 - rare ionocyte
        'Transitioning',        # 24 - mixed/low quality
        'Metaplastic',      # 25 - squamous metaplasia
    ],
    
    # ===== Level 3: Detailed Subtype =====
    'cell_type_level_4': [
        'Goblet_Progenitor_High_Metabolic',              # 0
        'Basal_to_Secretory_Differentiating',            # 1
        'Goblet_Mature_Immune_Active_PIGR_High',         # 2
        'Basal_Quiescent_Low_Expression',                # 3
        'Goblet_Mature_High_Secretory_Activity',         # 4
        'Basal_Proliferating_Cycling',                   # 5
        'Club_Mature_SCGB1A1_High',                      # 6
        'Goblet_Mature_Differentiated',                  # 7
        'Goblet_Mature_Moderate_Expression',             # 8
        'Goblet_Proliferating_Metabolic_Active',         # 9
        'Goblet_Classic_MUC5AC_Dominant',                # 10
        'Goblet_Hyperplastic_MUC2_Intestinal_Metaplasia',# 11
        'Secretory_Goblet_Hybrid_SCGB_MUC5B',            # 12
        'Club_to_Goblet_Transition_Early',               # 13
        'Goblet_Mature_Secretory_TFF3_PIGR_High',        # 14
        'Secretory_Proliferating_Club_Markers',          # 15
        'Goblet_Inflammatory_IFN_SAA_CSF3_Activated',    # 16
        'Goblet_Mature_PIGR_Secretory',                  # 17
        'Goblet_Differentiating_Early_Maturing',         # 18
        'Basal_Quiescent_Minimal_Expression',            # 19
        'Basal_Proliferating_ATP_Enriched',              # 20
        'Goblet_Stressed_ER_XBP1_CXCL1_High',            # 21
        'Club_Goblet_Transition_CXCL1_Inflammatory',     # 22
        'Rare_Ionocyte_FOLR1_Specialized',               # 23
        'Mixed_Uncertain_Quality',                       # 24
        'Goblet_Squamous_Metaplasia_SAA_Very_High',      # 25
    ],
    
    # Key marker expression
    'MUC5AC_mean': [1.88, 1.01, 2.20, 0.37, 2.04, 0.31, 1.27, 1.94, 1.29, 1.04,
                    2.28, 3.28, 1.58, 1.22, 2.31, 0.31, 1.47, 1.49, 1.31, 0.20,
                    1.02, 1.57, 1.67, 0.42, 0.76, 1.36],
    
    'MUC2_mean': [0.00, 0.11, 0.25, 0.00, 0.57, 0.00, 0.00, 0.19, 0.19, 0.58,
                  0.24, 1.88, 0.14, 0.00, 0.62, 0.01, 0.63, 0.63, 0.00, 0.01,
                  0.49, 0.09, 0.00, 0.02, 0.03, 1.58],
    
    'SCGB1A1_mean': [1.02, 0.80, 0.84, 0.56, 0.64, 0.84, 2.76, 0.45, 0.34, 1.03,
                     1.48, 0.49, 2.80, 2.96, 1.63, 2.38, 0.96, 2.12, 1.15, 0.47,
                     1.66, 0.94, 3.02, 0.90, 2.94, 0.10],
    
    'PIGR_mean': [2.37, 2.07, 2.85, 1.96, 2.52, 1.20, 2.34, 2.45, 2.23, 2.20,
                  2.71, 2.90, 2.54, 2.43, 2.91, 1.88, 2.11, 2.58, 1.56, 0.75,
                  2.64, 2.58, 2.56, 2.59, 2.15, 2.57],
    
    # Pathological features
    'Intestinal_Metaplasia': ['No', 'No', 'No', 'No', 'Mild', 'No', 'No', 'No', 'No', 'Mild',
                              'No', 'Yes', 'No', 'No', 'Mild', 'No', 'Mild', 'Mild', 'No', 'No',
                              'Mild', 'No', 'No', 'No', 'No', 'Yes'],
    
    'Inflammatory_State': ['Low', 'Low', 'Low', 'Low', 'Moderate', 'Low', 'Low', 'Low', 'Low', 'Low',
                          'Low', 'High', 'Moderate', 'Low', 'Low', 'Low', 'Very_High', 'Low', 'Low', 'Low',
                          'Low', 'High', 'High', 'Low', 'Low', 'Very_High'],
    
    'Annotation_Confidence': ['High', 'Medium', 'High', 'High', 'High', 'High', 'High', 'High', 'Medium', 'Medium',
                             'High', 'High', 'High', 'High', 'High', 'High', 'High', 'Medium', 'Medium', 'High',
                             'Medium', 'High', 'High', 'Medium', 'Low', 'High'],
    
    # Brief description
    'Description': [
        'Proliferating goblet progenitor with high metabolic activity',
        'Basal cells actively differentiating toward secretory/goblet lineage',
        'Mature goblet cells with strong immune function (PIGR-mediated IgA)',
        'Quiescent basal cells with minimal differentiation markers',
        'Mature goblet cells with high secretory activity and function',
        'Actively cycling basal cells with ATP/metabolic enrichment',
        'Mature club secretory cells (SCGB1A1 dominant)',
        'Well-differentiated mature goblet cells',
        'Mature goblet cells with moderate marker expression',
        'Proliferating goblet cells with high metabolic activity',
        'Classic mature respiratory goblet cells (MUC5AC dominant)',
        'Hyperplastic goblet with intestinal metaplasia (MUC2 positive)',
        'Transitioning hybrid cells (SCGB1A1+ MUC5B+ secretory-goblet)',
        'Early stage Club to Goblet transition',
        'Mature secretory goblet cells (TFF3/PIGR/SLPI high)',
        'Proliferating secretory cells retaining Club markers',
        'Inflammatory activated goblet (interferon response, SAA/CSF3 very high)',
        'Mature goblet cells with moderate secretory markers',
        'Early differentiating goblet cells in maturation process',
        'Quiescent basal cells with very low marker expression',
        'Proliferating basal/progenitor cells with ATP enrichment',
        'Stressed goblet cells (ER stress XBP1, inflammatory CXCL1)',
        'Club-Goblet transition driven by inflammation (CXCL1 high)',
        'Rare specialized cells (ionocytes, FOLR1 positive)',
        'Mixed/uncertain cell identity, possibly low quality cells',
        'Goblet cells with squamous metaplasia features (SAA extremely high)'
    ]
}

# Create DataFrame
annotation_df = pd.DataFrame(annotation_data)

# Save to CSV
annotation_df.to_csv(f"{MARKER_TABLE_DIR}/goblet_complete_annotation_3levels.csv", index=False)

print("=" * 80)
print("COMPLETE ANNOTATION SYSTEM (3 LEVELS - REVISED)")
print("=" * 80)
print(annotation_df[['cluster', 'cell_type_level_2', 'cell_type_level_3', 
                     'cell_type_level_4', 'MUC5AC_mean', 'MUC2_mean']].to_string(index=False))
print("\n" + "=" * 80)
print(f"✓ Saved: {MARKER_TABLE_DIR}/goblet_complete_annotation_3levels.csv")
print("=" * 80)

# ===== Summary Statistics =====
print("\n" + "=" * 80)
print("ANNOTATION SUMMARY")
print("=" * 80)

print("\n【Level 1】Major Lineage Distribution:")
level1_counts = annotation_df['cell_type_level_2'].value_counts()
for lineage, count in level1_counts.items():
    print(f"  {lineage:15s}: {count:2d} clusters ({count/26*100:5.1f}%)")

print("\n【Level 2】Functional State Distribution:")
level2_counts = annotation_df['cell_type_level_3'].value_counts()
for state, count in level2_counts.items():
    clusters = annotation_df[annotation_df['cell_type_level_3']==state]['cluster'].tolist()
    print(f"  {state:20s}: {count:2d} clusters - C{','.join(clusters)}")

print("\n【Pathological Features】")
meta_yes = (annotation_df['Intestinal_Metaplasia'] == 'Yes').sum()
meta_mild = (annotation_df['Intestinal_Metaplasia'] == 'Mild').sum()
inflam_high = (annotation_df['Inflammatory_State'].isin(['High', 'Very_High'])).sum()
print(f"  Intestinal Metaplasia (Severe): {meta_yes} clusters - C{','.join(annotation_df[annotation_df['Intestinal_Metaplasia']=='Yes']['cluster'].tolist())}")
print(f"  Intestinal Metaplasia (Mild):   {meta_mild} clusters")
print(f"  High Inflammatory State:        {inflam_high} clusters")

# ===== Generate Mapping Files =====
print("\n" + "=" * 80)
print("GENERATING MAPPING FILES")
print("=" * 80)

# Python mapping
mapping_dict = annotation_df.set_index('cluster')[
    ['cell_type_level_2', 'cell_type_level_3', 'cell_type_level_4']
].to_dict('index')

with open(f"{MARKER_TABLE_DIR}/annotation_mapping.py", 'w') as f:
    f.write("# Goblet Cell Three-Level Annotation Mapping\n")
    f.write("# Generated: 2025-01-06\n\n")
    f.write("# Usage in Python/Scanpy:\n")
    f.write("#   adata.obs['cell_type_level_2'] = adata.obs['leiden'].astype(str).map(level_1_mapping)\n")
    f.write("#   adata.obs['cell_type_level_3'] = adata.obs['leiden'].astype(str).map(level_2_mapping)\n")
    f.write("#   adata.obs['cell_type_level_4'] = adata.obs['leiden'].astype(str).map(level_3_mapping)\n\n")
    
    f.write("# Level 1: Major Lineage (Basal/Secretory/Goblet/Rare/Mixed)\n")
    f.write("level_1_mapping = {\n")
    for cluster, info in mapping_dict.items():
        f.write(f"    '{cluster}': '{info['cell_type_level_2']}',\n")
    f.write("}\n\n")
    
    f.write("# Level 2: Functional State (Mature/Proliferating/Transitioning/etc.)\n")
    f.write("level_2_mapping = {\n")
    for cluster, info in mapping_dict.items():
        f.write(f"    '{cluster}': '{info['cell_type_level_3']}',\n")
    f.write("}\n\n")
    
    f.write("# Level 3: Detailed Subtype\n")
    f.write("level_3_mapping = {\n")
    for cluster, info in mapping_dict.items():
        f.write(f"    '{cluster}': '{info['cell_type_level_4']}',\n")
    f.write("}\n")

print(f"✓ Saved: {MARKER_TABLE_DIR}/annotation_mapping.py")

# R mapping
with open(f"{MARKER_TABLE_DIR}/annotation_mapping.R", 'w') as f:
    f.write("# Goblet Cell Three-Level Annotation Mapping\n")
    f.write("# Generated: 2025-01-06\n\n")
    f.write("# Usage in R/Seurat:\n")
    f.write("#   seurat_obj$cell_type_level_2 <- level_1_mapping[as.character(seurat_obj$seurat_clusters)]\n")
    f.write("#   seurat_obj$cell_type_level_3 <- level_2_mapping[as.character(seurat_obj$seurat_clusters)]\n")
    f.write("#   seurat_obj$cell_type_level_4 <- level_3_mapping[as.character(seurat_obj$seurat_clusters)]\n\n")
    
    f.write("# Level 1: Major Lineage\n")
    f.write("level_1_mapping <- c(\n")
    items = [f"  '{c}' = '{info['cell_type_level_2']}'" for c, info in mapping_dict.items()]
    f.write(",\n".join(items))
    f.write("\n)\n\n")
    
    f.write("# Level 2: Functional State\n")
    f.write("level_2_mapping <- c(\n")
    items = [f"  '{c}' = '{info['cell_type_level_3']}'" for c, info in mapping_dict.items()]
    f.write(",\n".join(items))
    f.write("\n)\n\n")
    
    f.write("# Level 3: Detailed Subtype\n")
    f.write("level_3_mapping <- c(\n")
    items = [f"  '{c}' = '{info['cell_type_level_4']}'" for c, info in mapping_dict.items()]
    f.write(",\n".join(items))
    f.write("\n)\n")

print(f"✓ Saved: {MARKER_TABLE_DIR}/annotation_mapping.R")

# ===== Apply to AnnData =====
print("\n" + "=" * 80)
print("APPLYING TO ADATA OBJECT")
print("=" * 80)

try:
    cluster_to_l1 = annotation_df.set_index('cluster')['cell_type_level_2'].to_dict()
    cluster_to_l2 = annotation_df.set_index('cluster')['cell_type_level_3'].to_dict()
    cluster_to_l3 = annotation_df.set_index('cluster')['cell_type_level_4'].to_dict()
    
    adata.obs['cell_type_level_2'] = adata.obs['leiden'].astype(str).map(cluster_to_l1)
    adata.obs['cell_type_level_3'] = adata.obs['leiden'].astype(str).map(cluster_to_l2)
    adata.obs['cell_type_level_4'] = adata.obs['leiden'].astype(str).map(cluster_to_l3)
    
    print("✓ Annotations applied to adata.obs")
    
    print("\n【Cell counts by Level 1】")
    print(adata.obs['cell_type_level_2'].value_counts().to_string())
    
    print("\n【Cell counts by Level 2】")
    print(adata.obs['cell_type_level_3'].value_counts().to_string())
    
except Exception as e:
    print(f"⚠ Could not apply to adata: {e}")

print("\n" + "=" * 80)
print("✅ ANNOTATION COMPLETE!")
print("=" * 80)
print("\n📊 Summary:")
print(f"  • Total clusters: 26")
print(f"  • Level 1 categories: {annotation_df['cell_type_level_2'].nunique()}")
print(f"  • Level 2 categories: {annotation_df['cell_type_level_3'].nunique()}")
print(f"  • Level 3 subtypes: 26 (unique per cluster)")
print("\n🎯 Key functional states (Level 2):")
print("  • Mature: Most abundant (healthy functional goblet/secretory cells)")
print("  • Proliferating: Actively cycling progenitors")
print("  • Transitioning: Club → Goblet differentiation")
print("  • Inflammatory/Stressed/Hyperplastic/Metaplastic: Pathological states")
print("=" * 80)

In [ ]:

# UMAP by cluster
print(f"\nGenerating UMAP plots...")

fig, ax = plt.subplots(figsize=(8, 6))
sc.pl.umap(
    adata,
    color='cell_type_level_2',
    ax=ax,
    show=False,
    legend_loc='on data',
    legend_fontsize=8,
    size=UMAP_SIZE,
    title='Goblet Cell Clusters (BBKNN-integrated)'
)
plt.tight_layout()
# plt.savefig(fig_dir / f'01_umap_clusters.{FIGURE_FORMAT}', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print(f"  ✓ Saved: 01_umap_clusters.{FIGURE_FORMAT}")
